# Farrell et al. (2020) on CRCNS ear-1: voltage sensing in the lateral membrane of outer hair cells

Paper. Farrell B, Skidmore BS, Rajasekharan V, Brownell WE (2020). A novel theoretical framework
reveals more than one voltage-sensing pathway in the lateral membrane of outer hair cells. J Gen
Physiol 152(7), doi:10.1085/jgp.201912447. The claims file
(`claims/crcns-ear-1/10.1085-jgp.201912447.yaml`) was written from the version of record
(PMC7335013) as flattened text: its displayed equations and Tables S1 to S4 are missing, and the
forms of the equations used here are the claims file's reconstructions.

Data set. CRCNS ear-1, doi:10.6080/K0571975, "Data collection curated with an application
ontology describes the methods and results upon performing an ex-vivo voltage-clamp assay on outer
hair cells of the mammalian cochlea" (Farrell and Bengtson), data description version 0.62 (May 11,
2020). Files are read from `ear-1/live/data_and_scripts.zip` in the public S3 bucket `crcnsarchive`,
streamed in place.

Version. CRCNS data sets are not versioned, so the version is recorded as `unversioned`
(docs/archives.md). The bucket holds two copies of the zip: `ear-1/live/` (MD5
9955733f5e7314a1f0983fbc18aab714 in the data set's `checksums.md5`, the file the data set's file
list names) and `ear-1/old/ver1/`. The paper cites the data set by its DOI only, with no version.
The live copy is the current published one; its description (v0.62) is the one that cites this
paper, so it is pinned here. The MD5 is read from `checksums.md5` below and printed, so that a later
change is detectable. The old copy was not read.

What the data set holds. 89 outer hair cells, one HDF5 file each (`HDF5_data/`), and the four
original MATLAB files they were translated from (`Original_Data/`, one per sex and maturity group),
with the per-cell values the authors derived: CL, QTm, V0.5 (raw and corrected), alphaB, Rm, Rs.
It does not hold CLW, CLn, the stereocilia capacitance CSB, the sequencing, the protein model, or
the values from other rodents.

| Claim | Tested or blocked |
|---|---|
| cohort, figs1-example-cell, nlw-capacitance, clw-by-turn, fig1-charge-not-linear, fig1-two-state-bounds | tested |
| alpha-pigmented-vs-albino, alpha-gradient-pigmented, table1-* (4), figs2-alphab-vs-clw | tested |
| v05-by-turn, fig2b-v05-vs-clw, fig2b-no-phenotype-difference, v05-gradient-length | tested |
| xe991-voltage-clamp, table2-* (4), v05-not-correlated-voltage-drop | tested |
| fig3a-eq3a-fit, fig3a-xe991-fit, fig3a-high-density-deviation, figs4-fine-bins, tables3-by-region | tested |
| capacitance-asymmetry, figs5-specific-capacitance, fig3b-negative-clw-fit, mean-slope-half | tested |
| fig4a-vh-rises-vd-constant, fig4d-cleq-two-limits, fig4c-sensor-area, fig4e-sensor-charge | tested |
| table3-pathway-energies, alternative-limit, low-chloride-similar, recording-time, two-hyperpolarized-states | tested |
| splice-isoform-x1, molecular-model | blocked: data missing |
| fig3a-other-rodents, fig3c-engineered-mice, fig4f-all-rodents, figs7-hearing-loss, v05-temperature-corrected | blocked: outside values not available here (`with_inputs`) |

CLW. The claims file's shared method was amended after a question to a reader
(`questions/crcns-ear-1.yaml`): CLW = CL - CNLW with CNLW = CSB + root determined per cell (Eqs. 8a
and 8c as rebuilt), judged under reading L-a (stored CLW, CNLW or CSB) where it exists, otherwise
L-b. The data set stores none of them, so L-b is used. The paper takes the stereocilia bundle
capacitance from Corbitt et al. (2012), doi:10.1016/j.bpj.2012.04.054, and that paper's rule is
used here: bundle area 0.07 um2 per um2 of membrane area AM = 2 pi r (L + r) up to 2000 um2, about
140 um2 above, at 0.0075 pF/um2 (Corbitt et al. 2012, Methods). AM is the stored cell surface area;
the 6 cells without one take the median CSB of their turn. The root is that of QTm against
ln(CL - CSB) over all 89 cells. The rule is an assumption (the paper does not give it), and it
passes the claims file's admissibility test: it gives the Fig. S1 cell CLW 18.30 pF, within 1.856
pF of the legend's 18.56 pF. Every outcome that uses CLW, dVmin or C_Leq rests on it. Reading L-c,
CLW = CL - 4.82 pF for every cell, is shown for comparison and decides nothing.

Groups outside the range of Eq. 3a. Under the CSB rule, the two smallest XE991 cells form a 5 pF
bin with mean dVmin 0.80 V (file 57 has CL 5.93 pF and CLW 0.48 pF), and one cell of Fig. 3b has a
negative CLWn. The paper's Fig. S4 fit includes only "Delta V min exp < 0.4 V", and its Discussion
assumes "V 0.5 is constant and independent for Delta V min exp > 0.3 V (Fig. 3 a and Fig. S4)". The
reading adopted here (y-1, an assumption) leaves bins or groups whose mean dVmin is not in (0, 0.4]
V out of every Eq. 3a fit and every group built on the Fig. 3a and Fig. S4 data (fig3a-xe991-fit,
fig3b-negative-clw-fit, mean-slope-half, fig4a-vh-rises-vd-constant, fig4d-cleq-two-limits and the
claims on its groups, low-chloride-similar). It leaves out no bin of the Fig. 3a fit without XE991. With that bin, the
XE991 fit has an adjusted R2 of -43, which the paper's 0.793 rules out, and the paper's smallest
sensor area (51 nm2) rules out its C_Leq of 0.20 aF (21 nm2). All bins (y-0) and the cells within
the range binned again (y-2) are shown as variants.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

import ear1

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

print("data set", ear1.DATASET, "version", ear1.VERSION)
md5 = ear1.zip_md5()
print("checksums.md5:", md5)
z = ear1.open_zip()
df, tr = ear1.load_cells(z)
mat = ear1.load_mat(z)
# df: reading L-b, CNLW = CSB + root per cell (ear1 docstring). dfB: reading L-c, CNLW = 4.82 pF
# for every cell, shown for comparison only (claims file, shared method).
dfB = ear1.with_cnlw(df, ear1.CNLW_C)


def within(x, v, tol):
    return bool(abs(x - v) <= tol + 1e-12)


def sets(d):
    """The cell sets the claims use: 144 mM chloride; and of those, without and with XE991."""
    c = d[d.chloride_mM == 144]
    return c, c[~c.xe991], c[c.xe991]


c144, no, xe = sets(df)
c144B, noB, xeB = sets(dfB)

In [ ]:
print("cells:", len(df), " files:", df.index.min(), "to", df.index.max())
print(pd.crosstab([df.phenotype, df.sex], df.maturity, margins=True))
print(pd.crosstab([df.colony, df.xe991], df.chloride_mM, margins=True))
print(pd.crosstab(df.turn.fillna(0).astype(int), df.polarity))
print("XE991 concentrations (uM):", df.xe991_uM.value_counts(dropna=False).to_dict())
print("temperature (C):", df.temperature_C.unique(), " pipette pressure (kPa):", df.pressure_kPa.unique())
print("f1 (Hz):", sorted(df.f1_Hz.unique()), " f2/f1:", np.unique(np.round(df.f2_Hz / df.f1_Hz, 6)))
units = {"CL": "pF", "QTm": "pC", "V05_raw": "V", "V05": "V", "alphaB": "1/V", "Rm": "MOhm",
         "Rs": "MOhm", "VD": "Rm/(Rm+Rs)", "t_after_death_min": "min", "length_um": "um",
         "lw_length_um": "um", "diameter_um": "um", "area_um2": "um2", "CLW": "pF", "dVmin": "V", "z": "e0"}
rng = df[list(units)].agg(["count", "min", "median", "max"]).T
rng["unit"] = pd.Series(units)
print(rng.round(4))

# Original MATLAB files against the HDF5 translation, matched on the stored CL.
m = mat.merge(df.reset_index()[["file", "CL", "orig", "turn", "xe991", "t_after_death_min", "V05",
                                "chloride_mM"]], on="CL", how="outer", indicator=True)
print("MAT-HDF5 match on CL:", m._merge.value_counts().to_dict())
print("cell numbers agree:", (m.cellnumber == m.orig).all(), " turn agrees:",
      (m.turn_x == m.turn_y.fillna(0)).all(), " time after death agrees:",
      (m.dT == m.t_after_death_min).all(), " corrected V0.5 agrees:", np.allclose(m.Vmcorr, m.V05))
print(pd.crosstab(m.toxins, m.xe991))

# Recordings deposited twice: identical capacitance traces.
dups = [(a, b) for i, a in enumerate(df.index) for b in df.index[i + 1:]
        if len(tr[a]["C"]) == len(tr[b]["C"]) and np.allclose(tr[a]["C"], tr[b]["C"])]
print("identical capacitance traces:", dups)
rep = df[df.orig.duplicated(keep=False)].sort_values("orig")[["orig", "researcher", "date", "phenotype"]]
print("original numbers used twice:", rep.orig.nunique(), "numbers,", len(rep), "cells; distinct dates within each:",
      rep.groupby("orig").date.nunique().min() == 2)

# Stored per-cell values against the traces.
re_ = pd.DataFrame({f: ear1.recompute_cell(tr[f], df.loc[f, "CL"]) for f in df.index}).T
print("QTm recomputed from stored CNL: max |diff| =", float((re_.QTm_re - df.QTm).abs().max()), "pC")
print("alphaB recomputed from stored q: max |diff| =", float((re_.alphaB_re - df.alphaB).abs().max()), "1/V")
print("corrected V0.5 = raw V0.5 x Rm/(Rm+Rs): max |diff| =", float((df.V05_raw * df.VD - df.V05).abs().max()), "V")
print("VD = Rm/(Rm+Rs) from the stored means: max |diff| =", float((df.Rm / (df.Rm + df.Rs) - df.VD).abs().max()))

# The trace and the stored V0.5 share the voltage axis: the voltage of the largest C on the ramp
# against the stored uncorrected V0.5.
vpk, vpk2 = {}, {}
for f in df.index:
    i0, i1 = ear1.ramp_indices(tr[f]["V"])
    v = tr[f]["V"][i0:i1 + 1]
    keep = v < 0.08
    vpk[f] = v[keep][np.argmax(tr[f]["C"][i0:i1 + 1][keep])]
    vpk2[f] = v[keep][np.argmax(tr[f]["C2"][i0:i1 + 1][keep])]
dv = pd.Series(vpk) - df.V05_raw
dv2 = pd.Series(vpk2) - df.V05_raw
near = (dv.abs() <= 0.0031 + 1e-9) | (dv2.abs() <= 0.0031 + 1e-9)
print("voltage of largest C (f1) minus stored raw V0.5 (V): median", round(float(dv.median()), 4),
      "; within one ramp step (0.0031 V) at f1:", int((dv.abs() <= 0.0031 + 1e-9).sum()), ", at f1 or f2:", int(near.sum()), "of", len(dv),
      "; largest |diff| of the others:", round(float(np.minimum(dv.abs(), dv2.abs())[~near].max()), 4) if (~near).any() else None)
hold = {f: float(np.median(tr[f]["V"][:5])) for f in df.index}
print("holding potential before the ramp (V):", pd.Series(hold).round(3).value_counts(dropna=False).to_dict(),
      "; files with NaN voltages:", [f for f in df.index if np.isnan(tr[f]["V"]).any()])

# The I-V arrays: the HDF5 layout against the original MATLAB files.
iv = [f for f in df.index if ear1.iv_table(tr[f]) is not None]
print("cells with DC I-V steps:", len(iv), " of which with XE991:", int(df.loc[iv, "xe991"].sum()))
t22 = tr[iv[0]]["IV_I"]
mrow = mat[np.isclose(mat.CL, df.loc[iv[0], "CL"])].iloc[0]
print("file", iv[0], ": HDF5 I_mean shape", np.shape(t22), ", first row as stored", np.round(t22[0], 2),
      "; MATLAB I_mean first row (mean, SD)", np.round(mrow.IV_I_mean[0], 2), "second row", np.round(mrow.IV_I_mean[1], 2))
same = all(np.allclose(ear1.iv_table(tr[f])[["I_pA", "I_sd_pA"]].to_numpy(),
                       mat[np.isclose(mat.CL, df.loc[f, "CL"])].iloc[0].IV_I_mean) for f in iv)
print("HDF5 I_mean reshaped to (2, N) equals the MATLAB (mean, SD) columns in all", len(iv), "cells:", same)

# Surface area: which formula the stored values follow.
r = df.diameter_um / 2
fa = pd.DataFrame({"2 pi r l": np.isclose(2 * np.pi * r * df.length_um, df.area_um2, rtol=1e-3),
                   "2 pi r LW + 2 pi r^2": np.isclose(2 * np.pi * r * df.lw_length_um + 2 * np.pi * r**2,
                                                      df.area_um2, rtol=1e-3)}, index=df.index)
print("cells with area:", int(df.area_um2.notna().sum()), " formula matches:", fa.sum().to_dict(),
      " neither:", int((df.area_um2.notna() & ~fa.any(axis=1)).sum()))
print("Rs < 10 MOhm:", int((df.Rs < 10).sum()), " Rs < 12 MOhm:", int((df.Rs < 12).sum()), "of", len(df))
# Reading L-b of CLW: CSB by the rule of Corbitt et al. (2012) on the stored area, root of QTm on
# ln(CL - CSB) over all 89 cells.
print("CSB (pF): range", round(df.CSB.min(), 3), "to", round(df.CSB.max(), 3), "; median", round(df.CSB.median(), 3),
      "; at the 1.05 pF plateau:", int((df.CSB >= 1.05 - 1e-9).sum()), "; imputed by turn (no area):",
      df.index[df.CSB_imputed].tolist())
print("root", round(df.attrs["root"], 3), "pF; CNLW = CSB + root: range", round(df.CNLW.min(), 3), "to",
      round(df.CNLW.max(), 3), "mean", round(df.CNLW.mean(), 3), "pF")
print("CLW (L-b): range", round(df.CLW.min(), 2), "to", round(df.CLW.max(), 2),
      "pF; CLW = CL - 4.82 pF (L-c):", round(dfB.CLW.min(), 2), "to", round(dfB.CLW.max(), 2))
print("CLW L-c minus L-b (pF): range", round((dfB.CLW - df.CLW).min(), 3), "to", round((dfB.CLW - df.CLW).max(), 3))
d_ = df[~df.CSB_imputed]
print("root without the", int(df.CSB_imputed.sum()), "cells whose CSB is imputed:", round(ear1.nlw_fit(d_.CL - d_.CSB, d_.QTm)["root"], 3), "pF (all 89:",
      round(df.attrs["root"], 3), "pF)")
print("I-V protocol of file", iv[0], ": steps (V)", np.round(tr[iv[0]]["IV_steps"], 3).tolist())
print("  mean current (pA)", ear1.iv_table(tr[iv[0]]).I_pA.round(1).tolist())
s_ = ear1.iv_table(tr[iv[0]]).iloc[1:-1].sort_values("V")
print("  chord conductance between neighbouring steps (nS): median", round(float(np.median(np.diff(s_.I_pA) / np.diff(s_.V) * 1e-3)), 2))

Checks of the description against the data:

- 89 cells, as the description says; each HDF5 file matches one record of the MATLAB files on CL,
  cell number, turn, time after death and corrected V0.5. No capacitance trace is deposited
  twice. Thirteen original cell numbers are used twice, each time by cells from different dates
  (and mostly different researchers and colonies), so the original number is not a unique
  identifier.
- Temperature 24 C and pipette pressure 0 kPa for every cell, as the Methods say. 144 mM chloride
  (140 mM CsCl + 2 mM MgCl2) in 86 cells and 10 mM (with 133 mM Cs glutamate) in 3, as the Methods
  say.
- XE991 is stored as 60 uM for 20 pigmented cells. The MATLAB `toxins` field labels 15 of them
  `XE991` and 5 `yes`. The Methods give 30 to 60 uM; no cell is stored with another concentration.
- The stored QTm, alphaB and corrected V0.5 follow from the stored traces and means as the data
  set's analysis script computes them (differences printed above). The largest capacitance on the
  ramp lies within one ramp step (0.0031 V) of the stored raw V0.5 in 79 of 89 cells, at the f1
  or the f2 trace (the script lets the user pick the peak on either), and within 0.0092 V in the
  rest, so the stored values and the traces share the voltage axis. This is consistent with the
  description; it does not show how the linear region was chosen, which the script leaves to the
  user. Five files pad their traces with NaN at the ends.
- One cell (file 52) holds at +0.06 V before and after the ramp, where the stimulus description
  says -0.06 V.
- The HDF5 `I_mean` of the I-V protocol has shape (N, 2) but stores its values in column order;
  read as given, the means and SDs are interleaved (a format error). It is reshaped here.
- The stored surface area (83 cells) follows two formulas: 2 pi r l for 34 cells with a cell
  length, and 2 pi r LW + 2 pi r^2 (a closed cylinder on the lateral wall length) for 50 (one cell
  fits both). The description does not say so.
- The stored surface area is used as AM for the stereocilia rule of Corbitt et al. (2012), which
  defines AM = 2 pi r (L + r) with L the lateral wall length. The 50 cells on the second formula
  match it exactly. The 34 cells on 2 pi r l match it only if the stored cell length l equals L + r,
  which the description does not say; the area recomputed as 2 pi r (L + r) from the lateral wall
  length gives the same root (5.100 against 5.104 pF, nlw-capacitance). The resulting CSB runs from
  0.35 to 1.05 pF (43 cells at the 1.05 pF plateau).
- Six cells have no stored area; their CSB is the median of their turn. Leaving them out moves the
  root from 5.104 to 5.137 pF.
- The I-V protocol steps are in V (-0.13 to 0.09 V in 0.01 V steps for file 22) and the mean
  currents in pA; the chord conductance between neighbouring steps is about 37 nS in that cell, so
  the DC conductances of the data set are in the nS range.

## cohort

> The data set holds the paper's 89 OHCs, 39 from adult males (17 albino, 22 pigmented), 21 from
> prepubertal males (15 albino, 6 pigmented), 28 from prepubertal females (10 albino, 18 pigmented)
> and 1 pigmented cell from an adult female.

The HDF5 maturity is `mature`, `Prepubertal` or `juvenile`; the MATLAB files put the `juvenile`
males in the prepubertal-male file, so `juvenile` is read as prepubertal.

In [ ]:
grp = df.assign(age=np.where(df.adult, "adult", "prepubertal"))
counts = grp.groupby(["age", "sex", "colony"]).size()
print(counts)
expected = {("adult", "male", "albino"): 17, ("adult", "male", "pigmented"): 22,
            ("prepubertal", "male", "albino"): 15, ("prepubertal", "male", "pigmented"): 6,
            ("prepubertal", "female", "albino"): 10, ("prepubertal", "female", "pigmented"): 18,
            ("adult", "female", "pigmented"): 1}
ok = {k: int(counts.get(k, 0)) == v for k, v in expected.items()}
print("group counts match:", ok, " albino adult females:", int(counts.get(("adult", "female", "albino"), 0)))
print("total", len(df), " albino", int((~df.pigmented).sum()), " pigmented", int(df.pigmented.sum()))
print("10 mM chloride:", int((df.chloride_mM == 10).sum()), " XE991:", int(df.xe991.sum()),
      " Rs < 10 MOhm:", int((df.Rs < 10).sum()), " recording time after death:",
      df.t_after_death_min.min(), "to", df.t_after_death_min.max(), "min")
print("MATLAB file of each group:", mat.groupby("mat_file").size().to_dict())

**Result: reproduced.** All eight group counts match: adult males 17 albino and 22 pigmented, prepubertal males 15 and 6, prepubertal females 10 and 18, and 1 pigmented adult female; 89 cells, 42 albino and 47 pigmented. As the paper says, 3 cells have 10 mM chloride, 81 have Rs < 10 MOhm, and recordings were made 47 to 182 min after death; 20 cells have XE991.

## figs1-example-cell

> The example cell of Fig. S1 (original number 99, juvenile pigmented female, 60 uM XE991) has CL
> 24.45 pF, CLn 25.94 pF, CLW 18.56 pF, V0.5 -0.022 V, QTm 3.02 pC and alphaB 34.98 V-1.

Recomputed from the trace: CL as the mean C over 0.09 to 0.15 V ("Values of membrane (Rm) and
series (Rs) resistances were determined from the electrical admittance at positive voltages
(typically 0.09-0.15 V) where there was no measurable voltage-dependent capacitance", Methods,
Whole-cell patch-clamp, paragraph 2); CNL = C - CL; QTm the sum of CNL dV over the ramp; alphaB
= 4 max(CNL)/QTm (the derivative of QNL/QTm is CNL/QTm); V0.5 the voltage of the largest C
(reading p-a) or of a fitted two-state Boltzmann derivative (p-b), times Rm/(Rm+Rs) averaged over
0.09 to 0.15 V. CLn by the readings of `ear1.cln_readings`.

In [ ]:
from scipy.optimize import curve_fit

cand = df[(df.orig == 99)]
print(cand[["orig", "phenotype", "sex", "maturity", "xe991_uM", "turn", "researcher", "date"]])
f99 = cand[cand.pigmented & cand.xe991].index[0]
t = tr[f99]
i0, i1 = ear1.ramp_indices(t["V"])
v, c = t["V"][i0:i1 + 1], t["C"][i0:i1 + 1]
rm, rs = t["Rm"][i0:i1 + 1], t["Rs"][i0:i1 + 1]
lin = (v >= 0.09) & (v <= 0.15)
CL_re = c[lin].mean()
cnl = c - CL_re
dvs = np.median(np.diff(v))
QTm_re = cnl.sum() * dvs
aB_re = 4 * cnl[v < 0.08].max() / QTm_re
vd_re = (rm[lin] / (rm[lin] + rs[lin])).mean()
V05_pa = v[v < 0.08][np.argmax(c[v < 0.08])] * vd_re


def boltz_c(vv, cl, q, a, vh):
    e = np.exp(-a * (vv - vh))
    return cl + q * a * e / (1 + e) ** 2


p, _ = curve_fit(boltz_c, v[v < 0.15], c[v < 0.15], p0=[CL_re, QTm_re, 30, -0.02])
V05_pb = p[3] * vd_re
cln = ear1.cln_readings(t["V"], t["C"], CL_re)
x = df.loc[f99]
tab = pd.DataFrame({
    "paper": [24.45, 25.94, 18.56, -0.022, 3.02, 34.98],
    "tolerance": [2.445, 2.594, 1.856, 0.0022, 0.302, 3.498],
    "recomputed": [CL_re, cln["first2"], CL_re - x.CNLW, V05_pa, QTm_re, aB_re],
    "stored": [x.CL, np.nan, x.CLW, x.V05, x.QTm, x.alphaB]},
    index=["CL pF", "CLn pF", "CLW pF", "V0.5 V", "QTm pC", "alphaB 1/V"])
tab["recomputed ok"] = [within(a, b, tl) for a, b, tl in zip(tab.recomputed, tab.paper, tab.tolerance)]
tab["stored ok"] = [within(a, b, tl) if np.isfinite(a) else None for a, b, tl in zip(tab.stored, tab.paper, tab.tolerance)]
print(tab.round(4))
print("V0.5 reading p-b (Boltzmann fit):", round(V05_pb, 4), "V;  CLn reading below -0.15 V:",
      round(cln["below_m015"], 2), "pF;  CLW with CNLW 4.82 pF (L-c):", round(CL_re - ear1.CNLW_C, 2), "pF")
print("L-b for this cell: area", round(x.area_um2, 1), "um2, CSB", round(x.CSB, 3), "pF, root", round(df.attrs["root"], 3),
      "pF, CNLW", round(x.CNLW, 3), "pF; stored CL - CNLW =", round(x.CLW, 2), "pF; L-b admissible (CLW within 18.56 +/- 1.856):",
      within(x.CLW, 18.56, 1.856), within(CL_re - x.CNLW, 18.56, 1.856))
print("recomputed values within tolerance:", int(tab["recomputed ok"].sum()), "of 6")

**Result: reproduced.** One cell has original number 99 and is pigmented, prepubertal, female and in 60 uM XE991. All six values recomputed from its trace are within tolerance: CL 24.58 pF, CLn 25.94 pF (the two most negative ramp points), CLW 18.43 pF, V0.5 -0.0216 V, QTm 2.92 pC, alphaB 36.04 V-1. The stored CL, V0.5, QTm and alphaB (24.45 pF, -0.0218 V, 3.02 pC, 34.99 V-1) are the legend's values. CLW is CL - CNLW under reading L-b: the cell's stored area is 2223 um2, so CSB is 1.05 pF, and with the root of 5.104 pF its CNLW is 6.154 pF, close to the 5.89 pF the legend implies. The stored CL then gives CLW 18.30 pF, also within 18.56 +/- 1.856 pF, so the CSB rule is admissible under the claims file's test. CL - 4.82 pF (L-c) would give 19.76 pF.

## nlw-capacitance

> The capacitance of the non-lateral-wall region is about 4.82 pF, close to the 4.86 pF reported
> before for adult males, and it does not differ between phenotypes or between adult and juvenile
> animals.

QTm = gamma + beta ln(CL - CSB), root exp(-gamma/beta), CNLW = CSB + root. The data set stores
neither CSB nor CNLW nor CLW, so readings (n-a) and (n-b) cannot be run as written. Run here are
(n-c), CSB = 0, and the CSB of Corbitt et al. (2012) (the rule of reading L-b, see the header), on
the stored area and on 2 pi r (L + r) recomputed from the diameter and lateral wall length; the
Corbitt CSB is computed, not stored, and stands in for n-a as an assumption. Each is read as (n-ii)
the root alone and (n-i) the mean per-cell CNLW, on all 89 cells and on the 86 with 144 mM
chloride. Intervals are 95% bootstrap intervals (2000 resamples of cells, seed fixed; a standard
choice).

In [ ]:
rng_nlw = np.random.default_rng(1)
# CSB readings: n-c, CSB = 0; "Corbitt stored area", the rule of Corbitt et al. (2012) on the stored
# cell surface area (the L-b reading of CLW; the data set stores no CSB, so this stands in for n-a);
# "Corbitt 2 pi r (L + r)", the same rule on the area recomputed from diameter and lateral wall length.
r_ = df.diameter_um / 2
CSBS = {"n-c CSB = 0": pd.Series(0.0, index=df.index), "Corbitt stored area": df.CSB,
        "Corbitt 2 pi r (L + r)": ear1.csb(df, 2 * np.pi * r_ * (df.lw_length_um + r_))[0]}


def nlw_root(s, cs, nboot=2000):
    x = s.CL - cs.loc[s.index]
    f = ear1.nlw_fit(x, s.QTm)
    bs = []
    idx = np.arange(len(s))
    for _ in range(nboot):
        k = rng_nlw.choice(idx, len(idx))
        bs.append(ear1.nlw_fit(x.iloc[k], s.QTm.iloc[k])["root"])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return f["root"], lo, hi, f["root"] + cs.loc[s.index].mean(), f["adj_r2"], len(s)


rows = {}
for cname, cs in CSBS.items():
    for name, s in [("all 89", df), ("144 mM", c144)]:
        rows[(cname, name, "all")] = nlw_root(s, cs)
        rows[(cname, name, "pigmented")] = nlw_root(s[s.pigmented], cs)
        rows[(cname, name, "albino")] = nlw_root(s[~s.pigmented], cs)
        rows[(cname, name, "adult")] = nlw_root(s[s.adult], cs)
        rows[(cname, name, "juvenile")] = nlw_root(s[~s.adult], cs)
nlw = pd.DataFrame(rows, index=["root pF (n-ii)", "ci lo", "ci hi", "mean CNLW pF (n-i)", "adj R2", "n"]).T
print(nlw.round(3).to_string())
for cname in CSBS:
    for name in ["all 89", "144 mM"]:
        t_ = nlw.sort_index().loc[(cname, name)]
        a = t_.loc["all"]
        ov1 = not (t_.loc["pigmented", "ci hi"] < t_.loc["albino", "ci lo"] or t_.loc["albino", "ci hi"] < t_.loc["pigmented", "ci lo"])
        ov2 = not (t_.loc["adult", "ci hi"] < t_.loc["juvenile", "ci lo"] or t_.loc["juvenile", "ci hi"] < t_.loc["adult", "ci lo"])
        print(cname, "|", name, ": root (n-ii)", round(a["root pF (n-ii)"], 3), "within 4.82 +/- 0.482:",
              within(a["root pF (n-ii)"], 4.82, 0.482), "; mean CNLW (n-i)", round(a["mean CNLW pF (n-i)"], 3), "within:",
              within(a["mean CNLW pF (n-i)"], 4.82, 0.482),
              "; pigmented-albino root difference", round(t_.loc["pigmented", "root pF (n-ii)"] - t_.loc["albino", "root pF (n-ii)"], 3),
              "overlap:", ov1, "; adult-juvenile", round(t_.loc["adult", "root pF (n-ii)"] - t_.loc["juvenile", "root pF (n-ii)"], 3),
              "overlap:", ov2)


**Result: reproduced (resting on the assumed CSB rule).** With the CSB of Corbitt et al. (2012) on the stored area, the root (n-ii) is 5.104 pF (95% interval 4.53 to 5.66) on all 89 cells and 5.125 pF on the 86 cells with 144 mM chloride, both within 4.82 +/- 0.482 pF; the recomputed area gives 5.100 and 5.120 pF. Read as the mean per-cell CNLW (n-i), the value is 5.98 pF, outside. The groups do not differ under this reading: the root intervals overlap for pigmented against albino (4.76 against 5.72 pF) and for adult against juvenile (5.26 against 4.96 pF). With CSB = 0 (n-c) the root is 5.44 pF (89 cells), outside tolerance, as before the claims file was amended.

## clw-by-turn

> The lateral membrane capacitance ranges from 5.0 (+/- 2.4) pF for turn 1 cells to 17.9 (+/- 2.74)
> pF for turn 4 cells.

In [ ]:
def clw_turn(d, label):
    c, n_, _ = sets(d)
    out = {}
    for rd, s in [("c-a all", d), ("c-b 144 mM", c), ("c-c Fig. 2 cells", n_)]:
        for tn, mu, sd in [(1, 5.0, 2.4), (4, 17.9, 2.74)]:
            g = s[s.turn == tn].CLW
            out[(label, rd, tn)] = {"n": len(g), "mean": g.mean(), "sd": g.std(), "mean ok": within(g.mean(), mu, sd),
                                    "sd within 10%": within(g.std(), sd, 0.1 * sd)}
    return pd.DataFrame(out).T


print(pd.concat([clw_turn(df, "L-b"), clw_turn(dfB, "L-c 4.82")]).round(3))
print("CLW by turn and colony (L-b, all cells):")
print(df.groupby(["turn", "colony"]).CLW.agg(["size", "mean", "std"]).round(2))

**Result: reproduced (resting on the assumed CSB rule).** Under L-b, mean CLW is 5.04 pF (SD 2.34, 8 cells) for turn 1 and 17.79 pF (SD 2.74, 36 cells) for turn 4 under readings c-a and c-b, and 5.15 and 18.32 pF for the Fig. 2 cells. All are within tolerance, and both SDs are within 10% of the paper's (2.4 and 2.74). CL - 4.82 pF (L-c) gives 5.85 and 19.10 pF.

## fig1-charge-not-linear

> The charge does not grow linearly with the lateral wall capacitance as identical and independent
> sensors would give (Eq. 6b fit, adjusted R2 0.516); the square of the charge rises much more
> steeply with CLW^2 for basal cells (CLW < 10 pF) than for apical cells.

The 86 cells with 144 mM chloride. For a fit through the origin the adjusted R2 is given with the
total sum of squares taken about the mean (centred, as MATLAB's curve fitting does) and about zero
(uncentred); the centred one is the first reading.

In [ ]:
def r2_origin(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    k = np.sum(x * y) / np.sum(x * x)
    sse = np.sum((y - k * x) ** 2)
    n = len(y)
    # Adjusted R2 with one coefficient: the residual has n - 1 degrees of freedom; the total sum of
    # squares has n - 1 about the mean (centred) and n about zero (uncentred, no mean estimated).
    cen = 1 - (sse / (n - 1)) / (np.sum((y - y.mean()) ** 2) / (n - 1))
    unc = 1 - (sse / (n - 1)) / (np.sum(y**2) / n)
    return k, cen, unc


def fig1(d, label):
    c, _, _ = sets(d)
    out = {}
    for scale, x, y in [("QTm^2 on CLW^2", c.CLW**2, c.QTm**2), ("QTm on CLW", c.CLW, c.QTm)]:
        k, cen, unc = r2_origin(x, y)
        wi = ear1.ols(x, y)
        out[(label, scale, "f-a origin, centred")] = (k, cen)
        out[(label, scale, "f-a origin, uncentred")] = (k, unc)
        out[(label, scale, "f-b intercept")] = (wi["slope"], wi["adj_r2"])
    t_ = pd.DataFrame(out, index=["slope", "adj R2"]).T
    t_["within 0.516 +/- 0.0516"] = [within(a, 0.516, 0.0516) for a in t_["adj R2"]]
    lo, hi = c[c.CLW < 10], c[c.CLW >= 10]
    sl = {}
    for ic in [False, True]:
        a = ear1.ols(lo.CLW**2, lo.QTm**2, intercept=ic)
        b = ear1.ols(hi.CLW**2, hi.QTm**2, intercept=ic)
        ci_a = (a["slope"] - 1.96 * a["se_slope"], a["slope"] + 1.96 * a["se_slope"])
        ci_b = (b["slope"] - 1.96 * b["se_slope"], b["slope"] + 1.96 * b["se_slope"])
        sl["intercept" if ic else "origin"] = {"n<10": a["n"], "slope<10": a["slope"], "ci<10": np.round(ci_a, 4),
                                               "n>=10": b["n"], "slope>=10": b["slope"], "ci>=10": np.round(ci_b, 4),
                                               "ratio": a["slope"] / b["slope"], "disjoint": ci_a[0] > ci_b[1]}
    return t_, pd.DataFrame(sl).T


for d, lab in [(df, "L-b"), (dfB, "L-c 4.82")]:
    a, b = fig1(d, lab)
    print(a.round(3))
    print(b)
c = c144
print("ranges plotted: CLW^2", round((c.CLW**2).min(), 1), "to", round((c.CLW**2).max(), 1), "pF^2; QTm^2",
      round((c.QTm**2).min(), 3), "to", round((c.QTm**2).max(), 3), "pC^2")
fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(c.CLW**2, c.QTm**2, c=np.where(c.CLW < 10, "C0", "C1"), s=12)
ax.set_xlabel("CLW$^2$ (pF$^2$)")
ax.set_ylabel("QTm$^2$ (pC$^2$)")
plt.show()

**Result: reproduced (resting on the assumed CSB rule).** Under L-b, the Eq. 6b fit through the origin on the scale of QTm^2 against CLW^2 has adjusted R2 0.539 (centred), within 0.516 +/- 0.052; the uncentred R2 is 0.891, with an intercept 0.687, and 0.64 to 0.96 on the scale of QTm against CLW. Under the same reading the square of the charge rises more steeply for cells with CLW < 10 pF: through the origin the slope of QTm^2 on CLW^2 is 0.0496 for 31 cells against 0.0248 for 55 cells (ratio 2.00, disjoint 95% intervals); with an intercept it is 0.0410 against 0.0091 (ratio 4.51). Under L-c the centred R2 is 0.585, outside.

## alpha-pigmented-vs-albino

> The maximum voltage sensitivity alpha(V0.5) is significantly greater for OHCs from pigmented
> animals (8.62 +/- 0.95 V-1, 28 cells) than from albinos (7.65 +/- 0.88 V-1, 41 cells).

Cells without XE991 with 144 mM chloride (the criterion's Table 1 rows b and c). The data set
gives 27 pigmented and 42 albino such cells; 41 albino cells have a known region, which is shown
as a variant.

In [ ]:
def alpha_colony(s, label):
    p, a = s[s.pigmented].alpha, s[~s.pigmented].alpha
    f, pa = ear1.anova1(p, a)
    w = stats.ttest_ind(p, a, equal_var=False)
    return {"set": label, "n pig": len(p), "mean pig": p.mean(), "sd pig": p.std(), "n alb": len(a),
            "mean alb": a.mean(), "sd alb": a.std(), "p t-a ANOVA": pa, "p t-b Welch": w.pvalue,
            "pig ok": within(p.mean(), 8.62, 0.95), "alb ok": within(a.mean(), 7.65, 0.88),
            "pig 10%": within(p.mean(), 8.62, 0.862), "alb 10%": within(a.mean(), 7.65, 0.765)}


print(pd.DataFrame([alpha_colony(no, "144 mM, no XE991"),
                    alpha_colony(no[no.region.notna()], "same, known region"),
                    alpha_colony(no[no.orig != 3], "same, without cell orig 3 (alphaB 57.5)")]).round(5).T)
print("by sex (pigmented):", no[no.pigmented].groupby("sex").alpha.agg(["size", "mean"]).round(3).to_dict())

**Result: reproduced.** alpha(V0.5) is 9.00 +/- 1.42 V-1 for 27 pigmented cells against 7.81 +/- 0.93 V-1 for 42 albino cells (one-way ANOVA p = 0.00008, Welch p = 0.0004). Both means are within the stated SDs and within 10%. Leaving out cell 3 (alphaB 57.5 V-1) gives 8.79 +/- 0.95 V-1 for the pigmented cells.

## alpha-gradient-pigmented, table1-pigmented-xe991, table1-pigmented-no-xe991, table1-albino-no-gradient

> For pigmented animals alpha(V0.5) falls from base (9.32 +/- 0.69 V-1) to apex (8.08 +/- 0.76
> V-1), a significant difference, while albinos show no gradient (basal 7.76 +/- 0.98, apical 7.59
> +/- 0.83 V-1).

> Table 1. With XE991 (pigmented): basal alphaB 36.4, z 0.93, 4 cells; apical 33.8, z 0.86, 11
> cells; P = 0.0227. Without XE991 (pigmented): basal 37.3, z 0.96, 12 cells; apical 32.3, z
> 0.83, 16 cells; P < 0.0001. Albino: basal 31.3, z 0.80, 11 cells; apical 30.5, z 0.78, 30
> cells; P = 0.5294.

One-way ANOVA of alphaB (or alpha) on region; regions by the data set's polarity label (r-a,
turns 1-2 against 3-4) and turn 1 against turn 4 (r-b).

In [ ]:
T1 = {"pig XE991": (36.4, 1.5, 33.8, 1.8, 0.93, 0.04, 0.86, 0.05, 4, 11),
      "pig no XE991": (37.3, 2.7, 32.3, 3.0, 0.96, 0.07, 0.83, 0.08, 12, 16),
      "albino": (31.3, 3.8, 30.5, 3.4, 0.80, 0.10, 0.78, 0.09, 11, 30)}


def table1(d):
    c, n_, x_ = sets(d)
    sel = {"pig XE991": x_[x_.pigmented], "pig no XE991": n_[n_.pigmented], "albino": n_[~n_.pigmented]}
    rows = []
    for g, s in sel.items():
        e = T1[g]
        for reg in ["region", "region_rb"]:
            b, a = s[s[reg] == "basal"], s[s[reg] == "apical"]
            f, p = ear1.anova1(b.alphaB, a.alphaB)
            rows.append({"group": g, "reading": "r-a" if reg == "region" else "r-b", "n basal": len(b), "n apical": len(a),
                         "aB basal": b.alphaB.mean(), "sd b": b.alphaB.std(), "aB apical": a.alphaB.mean(),
                         "sd a": a.alphaB.std(), "z basal": b.z.mean(), "z apical": a.z.mean(), "p": p,
                         "means ok": all([within(b.alphaB.mean(), e[0], e[1]), within(a.alphaB.mean(), e[2], e[3]),
                                          within(b.z.mean(), e[4], e[5]), within(a.z.mean(), e[6], e[7])]),
                         "counts paper": f"{e[8]}/{e[9]}"})
    return pd.DataFrame(rows)


t1 = table1(df)
print(t1.round(4).to_string())
print("alpha = alphaB / 4 per group (for alpha-gradient-pigmented):")
for g, (bm, bs, am, as_) in {"pig no XE991": (9.32, 0.69, 8.08, 0.76), "albino": (7.76, 0.98, 7.59, 0.83)}.items():
    for _, r in t1[t1.group == g].iterrows():
        print(" ", g, r.reading, "basal", round(r["aB basal"] / 4, 3), "(ok", within(r["aB basal"] / 4, bm, bs), ") apical",
              round(r["aB apical"] / 4, 3), "(ok", within(r["aB apical"] / 4, am, as_), ") p", round(r.p, 5))
pb = no[no.pigmented & (no.region == "basal")]
print("pigmented basal without XE991, without cell orig 3: n", len(pb[pb.orig != 3]), "alphaB",
      round(pb[pb.orig != 3].alphaB.mean(), 2), "sd", round(pb[pb.orig != 3].alphaB.std(), 2))
print("Variant: pigmented cell orig 91 (the only XE991 cell recorded by T. Kinney and W. Abdelhafez) counted without XE991:")
alt = df.copy()
alt.loc[alt.orig.eq(91) & alt.pigmented, "xe991"] = False
print(table1(alt)[["group", "reading", "n basal", "n apical", "aB basal", "aB apical", "p", "means ok"]].round(4))

**Result: partially reproduced.** Under the data set's regions (r-a) the pigmented XE991 cells number 5 basal and 12 apical, where the paper has 4 and 11. alphaB is 37.49 against 33.85 V-1 and z 0.96 against 0.87, all within tolerance, with ANOVA p = 0.0027. Reading r-b leaves one basal cell. The direction and significance hold, but the counts differ.

**Result: partially reproduced.** Under r-a there are 12 basal and 15 apical cells (the paper has 12 and 16). alphaB is 39.73 +/- 6.04 against 32.99 +/- 3.17 V-1 and z 1.017 against 0.845, all within tolerance. The ANOVA gives p = 0.0010, not the paper's p < 0.0001; under r-b p = 0.019. The basal mean includes cell 3 (alphaB 57.5 V-1); without it, the 11 basal cells average 38.11 +/- 2.39 V-1. Counting cell 91 as without XE991 gives 12 and 16 cells, with p = 0.0008.

**Result: reproduced.** Under r-a there are 11 basal and 30 apical albino cells, the paper's counts. alphaB is 31.84 against 31.11 V-1 (difference 0.73 V-1) and z 0.815 against 0.797, all within tolerance. The ANOVA gives p = 0.586 (the paper has 0.5294), so the test finds no difference. Under r-b p = 0.977.

**Result: reproduced.** Under r-a, alpha(V0.5) of pigmented cells without XE991 is 9.93 V-1 basal against 8.25 V-1 apical (ANOVA p = 0.00097). For albino cells it is 7.96 against 7.78 V-1 (p = 0.586). All four means are within the stated SDs. Under r-b the values are 9.05 against 8.06 (p = 0.019) and 7.75 against 7.73 (p = 0.977).

## table1-xe991-no-effect-alpha

> Adding XE991 to the bath has no significant effect on alphaB (P = 0.7551), so pigmented cells
> with and without XE991 are pooled, giving basal alphaB 37.1 +/- 2.5 V-1 (z 0.95, 16 cells) and
> apical 32.9 +/- 2.7 V-1 (z 0.84, 27 cells).

Effect tests from an ordinary least squares model (type II sums of squares, which equal JMP's
effect tests for a model without interactions).

In [ ]:
def effect_tests(d, y):
    c = d[(d.chloride_mM == 144) & d.region.notna()]
    pig = c[c.pigmented]
    out = {}
    a = anova_lm(smf.ols(f"{y} ~ C(region) + C(xe991)", pig).fit(), typ=2)
    out["x-a two-way, pigmented"] = (a.loc["C(xe991)", "PR(>F)"], smf.ols(f"{y} ~ C(region) + C(xe991)", pig).fit().params["C(xe991)[T.True]"])
    mb = smf.ols(f"{y} ~ C(region) + C(xe991) + C(colony)", c).fit()
    out["x-b three factors, all"] = (anova_lm(mb, typ=2).loc["C(xe991)", "PR(>F)"], mb.params["C(xe991)[T.True]"])
    mc = smf.ols(f"{y} ~ C(xe991)", pig).fit()
    out["x-c one-way, pigmented"] = (anova_lm(mc, typ=2).loc["C(xe991)", "PR(>F)"], mc.params["C(xe991)[T.True]"])
    return pd.DataFrame(out, index=["p XE991", "XE991 coefficient"]).T


print(effect_tests(df, "alphaB").round(4))
pig = c144[c144.pigmented & c144.region.notna()]
pool = pig.groupby("region").agg(n=("alphaB", "size"), alphaB=("alphaB", "mean"), sd=("alphaB", "std"),
                                 z=("z", "mean"), z_sd=("z", "std"))
print(pool.round(3))
print("basal within 37.1 +/- 2.5:", within(pool.loc["basal", "alphaB"], 37.1, 2.5), " apical within 32.9 +/- 2.7:",
      within(pool.loc["apical", "alphaB"], 32.9, 2.7), " z basal within 0.95 +/- 0.06:",
      within(pool.loc["basal", "z"], 0.95, 0.06), " z apical within 0.84 +/- 0.07:", within(pool.loc["apical", "z"], 0.84, 0.07))

**Result: partially reproduced.** The effect test finds no XE991 effect on alphaB: p = 0.86 (x-a), 0.66 (x-b) and 0.47 (x-c), with coefficients of -0.21 to -1.06 V-1. The pooled pigmented means are basal 39.07 +/- 5.24 V-1 (z 1.00) and apical 33.38 +/- 2.63 V-1 (z 0.855), all within tolerance. The counts are 17 and 27, where the paper has 16 and 27.

## figs2-alphab-vs-clw

> alphaB decreases monotonically with lateral membrane capacitance for cells from pigmented
> animals, with no relationship for cells from albinos.

In [ ]:
rows = []
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, _ = sets(d)
    for cells, s in [("no XE991", n_), ("with XE991 cells", c)]:
        for col in ["pigmented", "albino"]:
            g = s[s.colony == col]
            sp = stats.spearmanr(g.CLW, g.alphaB)
            ls = ear1.ols(g.CLW, g.alphaB)
            rows.append({"CLW": lab, "cells": cells, "colony": col, "n": len(g), "spearman rho": sp.statistic,
                         "p s-a": sp.pvalue, "slope 1/V per pF": ls["slope"], "p s-b": ls["p_slope"]})
print(pd.DataFrame(rows).round(4))

**Result: reproduced (resting on the assumed tests).** For pigmented cells without XE991, alphaB falls with CLW (Spearman rho -0.85, p < 0.0001; slope -0.59 V-1 per pF under L-b). For albino cells there is no relation (rho -0.17, p = 0.29; slope p = 0.75). With the XE991 cells included, the pigmented rho is -0.82. L-c gives the same picture (slope -0.57 V-1 per pF).

## v05-by-turn

> V0.5 is not constant but ranges from -0.083 +/- 0.030 V (turn 1) to -0.050 +/- 0.018 V (turn 4)
> for albino cells, and from -0.102 +/- 0.019 V to -0.052 +/- 0.020 V for pigmented cells.

In [ ]:
exp_v = {("albino", 1): (-0.083, 0.030), ("albino", 4): (-0.050, 0.018),
         ("pigmented", 1): (-0.102, 0.019), ("pigmented", 4): (-0.052, 0.020)}
rows = []
for rd, s in [("v-a Fig. 2 cells", no), ("v-b all cells", df)]:
    for (col, tn), (mu, sd) in exp_v.items():
        g = s[(s.colony == col) & (s.turn == tn)].V05
        rows.append({"reading": rd, "colony": col, "turn": tn, "n": len(g), "mean": g.mean(), "sd": g.std(),
                     "within sd": within(g.mean(), mu, sd), "within 10%": within(g.mean(), mu, abs(0.1 * mu))})
print(pd.DataFrame(rows).round(4))
print(no.groupby(["colony", "turn"]).V05.agg(["size", "mean", "std"]).round(4))

**Result: reproduced.** In the Fig. 2 cells (v-a), V0.5 is -0.0826 +/- 0.0299 V (3 cells) for albino turn 1 and -0.0496 +/- 0.0176 V (15) for turn 4. For pigmented cells it is -0.1025 +/- 0.0190 V (4) for turn 1 and -0.0524 +/- 0.0201 V (11) for turn 4. All four are within the stated SDs and within 10%, and turn 1 is more negative in both colonies. Under v-b, the pigmented turn 4 mean (-0.0448 V, 21 cells with XE991 included) is within the SD but not within 10%.

## fig2b-v05-vs-clw and fig2b-no-phenotype-difference

> V0.5 increases monotonically with CLW, becoming more positive as CLW increases; the linear
> relation is better for pigmented cells (adjusted R2 about 0.6) than for albino cells (about 0.2).

> The V0.5 gradient with CLW does not differ statistically between the two phenotypes.

The cells of Fig. 2: 144 mM chloride, no XE991 (69 in the data set, 70 in the legend).

In [ ]:
for lab, s in [("L-b", no), ("L-c 4.82", noB)]:
    res = {}
    for col, e in [("pigmented", 0.6), ("albino", 0.2)]:
        g = s[s.colony == col]
        r = ear1.ols(g.CLW, g.V05)
        res[col] = {"n": r["n"], "slope V/pF": r["slope"], "se": r["se_slope"], "p": r["p_slope"],
                    "adj R2": r["adj_r2"], "R2 ok": within(r["adj_r2"], e, 0.1 * e)}
    print(lab)
    print(pd.DataFrame(res).T.round(5))
    m_ = smf.ols("V05 ~ CLW * C(colony)", s).fit()
    pi = m_.pvalues["CLW:C(colony)[T.pigmented]"]
    d_ = res["pigmented"]["slope V/pF"] - res["albino"]["slope V/pF"]
    se = np.hypot(res["pigmented"]["se"], res["albino"]["se"])
    dfree = len(s) - 4
    print("  g-a interaction p =", round(pi, 4), "; g-b slope difference", round(d_, 5), "V/pF, t =", round(d_ / se, 3),
          "p =", round(2 * stats.t.sf(abs(d_ / se), dfree), 4))
print("ranges: CLW", round(no.CLW.min(), 2), "to", round(no.CLW.max(), 2), "pF; V0.5", round(no.V05.min(), 4), "to", round(no.V05.max(), 4), "V")

**Result: partially reproduced.** V0.5 rises with CLW in both colonies. Under L-b, for pigmented cells the slope is 0.00335 V/pF (p < 0.0001) with adjusted R2 0.613, within tolerance. For albino cells it is 0.00173 V/pF (p = 0.0034) with adjusted R2 0.175, outside 0.2 +/- 0.02. The fit is better for pigmented cells, as stated. L-c gives 0.612 and 0.172. The data set has 69 such cells, where the legend counts 70.

**Result: not reproduced.** Under L-b the CLW by colony interaction is significant (p = 0.039). The pigmented slope (0.00335 V/pF) exceeds the albino slope (0.00173 V/pF) by 0.00162 V/pF; the slope t-test (g-b) gives t = 2.14, p = 0.037. L-c gives the same p values.

## v05-gradient-length

> In guinea pig OHCs the V0.5 gradient along the cochlea is 0.7 um/mV in terms of cell length.

The data set has two length fields: `cochlear_outer_hair_cell_lateral_wall_length` (83 cells) and
`cochlear_outer_hair_cell_length` (35 cells). Both are run.

In [ ]:
rows = []
for lcol in ["lw_length_um", "length_um"]:
    for cells, s in [("l-1 Fig. 2 cells", no), ("l-2 all cells", df)]:
        g = s[s[lcol].notna()]
        a = ear1.ols(g.V05 * 1000, g[lcol])
        b = ear1.ols(g[lcol], g.V05 * 1000)
        rows.append({"length": lcol, "cells": cells, "n": len(g), "l-a um/mV": a["slope"], "p": a["p_slope"],
                     "l-b um/mV": 1 / b["slope"], "l-a ok": within(a["slope"], 0.7, 0.07), "l-b ok": within(1 / b["slope"], 0.7, 0.07)})
print(pd.DataFrame(rows).round(4))
# Variant not in the claims file (asked of a reader, questions/crcns-ear-1.yaml): the end-to-end gradient,
# the difference in mean length between turns 4 and 1 over the difference in mean V0.5.
rows = []
for lcol in ["lw_length_um", "length_um"]:
    for cells, s in [("Fig. 2 cells", no), ("all cells", df)]:
        g = s[s[lcol].notna()]
        print(lcol, cells)
        print(g.groupby(["colony", "turn"]).agg(n=(lcol, "size"), length=(lcol, "mean"), V05=("V05", "mean")).round(4).to_string())
        for col in ["albino", "pigmented", "both"]:
            gg = g if col == "both" else g[g.colony == col]
            t1_, t4_ = gg[gg.turn == 1], gg[gg.turn == 4]
            if len(t1_) and len(t4_):
                dl, dv_ = t4_[lcol].mean() - t1_[lcol].mean(), 1000 * (t4_.V05.mean() - t1_.V05.mean())
                rows.append({"length": lcol, "cells": cells, "colony": col, "n turn 1": len(t1_), "n turn 4": len(t4_),
                             "dL um": dl, "dV0.5 mV": dv_, "um/mV": dl / dv_, "within 0.63-0.77": within(dl / dv_, 0.7, 0.07)})
print("end-to-end variant:")
print(pd.DataFrame(rows).round(3).to_string())

**Result: not reproduced.** Regressing length on V0.5 (l-a) gives 0.46 um/mV on the lateral wall length of 64 Fig. 2 cells, 0.45 um/mV on all 83 cells, and 0.50 um/mV on the cell length of 35 cells. The inverse slope (l-b) gives 1.37 to 1.58 um/mV. None is within 0.63 to 0.77 um/mV. The end-to-end gradient, the difference in mean length between turns 4 and 1 over the difference in mean V0.5, is not a reading of the claims file (it is asked of a reader in questions/crcns-ear-1.yaml); it gives 0.95 to 1.43 um/mV, also outside.

## xe991-voltage-clamp

> XE991 improves the voltage clamp at least twofold, the fractional pipette drop staying below 13%
> (against e.g. 26% at voltages above 0.08 V without), while conductance at V0.5 with XE991 reaches
> 335 pS.

Fractional drop Rs/(Rs+Rm): (d-b) at positive voltages from the stored means; (d-a) at V0.5 from
the admittance Rm and Rs at the ramp point nearest the raw V0.5, and, for the cells with DC steps,
with the DC chord conductance at V0.5. All 20 XE991 cells and the 69 cells without.

In [ ]:
dropV, gdc, gadm = {}, {}, {}
for f in df.index:
    t = tr[f]
    i0, i1 = ear1.ramp_indices(t["V"])
    k = i0 + int(np.argmin(np.abs(t["V"][i0:i1 + 1] - df.loc[f, "V05_raw"])))
    dropV[f] = t["Rs"][k] / (t["Rs"][k] + t["Rm"][k])
    gadm[f] = 1e3 / t["Rm"][k]                 # nS
    gdc[f] = ear1.dc_conductance_at(t, df.loc[f, "V05_raw"])
vc = df[["xe991", "drop_pos", "Rs"]].assign(drop_V05=pd.Series(dropV), G_adm_nS=pd.Series(gadm), G_dc_nS=pd.Series(gdc))
vc["drop_V05_dc"] = vc.Rs / (vc.Rs + 1e3 / vc.G_dc_nS)
print(vc.groupby("xe991")[["drop_pos", "drop_V05", "drop_V05_dc", "G_adm_nS", "G_dc_nS"]].agg(["count", "median", "mean", "max"]).round(4).T)
for col in ["drop_V05", "drop_pos"]:
    a, b = vc[vc.xe991][col], vc[~vc.xe991][col]
    print(col, ": XE991 cells below 0.143 (13% widened by 10%):", int((a < 0.143).sum()), "of", len(a),
          "; ratio of medians without/with", round(b.median() / a.median(), 3), "; ratio of means", round(b.mean() / a.mean(), 3))
print("largest DC conductance at V0.5 among XE991 cells:", round(vc[vc.xe991].G_dc_nS.max(), 3), "nS (", int(vc[vc.xe991].G_dc_nS.notna().sum()),
      "cells); largest admittance conductance 1/Rm at V0.5:", round(vc[vc.xe991].G_adm_nS.max(), 3), "nS; claim 0.335 nS")
print("cells without XE991 with a drop above 0.08 V greater than 0.26:", int((vc[~vc.xe991].drop_pos > 0.26).sum()), "; largest", round(vc[~vc.xe991].drop_pos.max(), 3))

**Result: not reproduced.** All 20 XE991 cells have a fractional drop below 13%: at most 0.065 at V0.5 and 0.127 at positive voltages. The twofold improvement does not hold by medians: the median without XE991 is 1.52 times the median with it at V0.5, and 1.51 times at positive voltages. By means it is 2.15 and 1.66 times. The largest conductance at V0.5 among XE991 cells is 8.53 nS (DC, 12 cells) or 7.02 nS (admittance 1/Rm), about 25 times the 335 pS claimed. Without XE991, the largest drop is 0.260 at positive voltages. The criterion needs all three conditions and has no partial tier; two of them fail.

## table2-pigmented-xe991, table2-pigmented-no-xe991, table2-albino, table2-xe991-v05-shift

> Table 2. With XE991 (pigmented): V0.5 basal -0.062 V (4 cells), apical -0.035 V (11), P =
> 0.0399. Without XE991 (pigmented): basal -0.091 V (12), apical -0.054 V (16), P = 0.0003.
> Albino: basal -0.071 V (11), apical -0.057 V (30), P = 0.0697.

> XE991 shifts V0.5 by about 20 mV in the depolarizing direction (ANOVA effect test P = 0.0035 in
> pigmented cells) without affecting the other electrical parameters.

In [ ]:
T2 = {"pig XE991": (-0.062, 0.020, -0.035, 0.020, 0.05), "pig no XE991": (-0.091, 0.022, -0.054, 0.023, 0.001),
      "albino": (-0.071, 0.024, -0.057, 0.019, None)}
sel = {"pig XE991": xe[xe.pigmented], "pig no XE991": no[no.pigmented], "albino": no[~no.pigmented]}
rows = []
for g, s in sel.items():
    e = T2[g]
    for reg in ["region", "region_rb"]:
        b, a = s[s[reg] == "basal"].V05, s[s[reg] == "apical"].V05
        f, p = ear1.anova1(b, a)
        rows.append({"group": g, "reading": "r-a" if reg == "region" else "r-b", "n basal": len(b), "n apical": len(a),
                     "V basal": b.mean(), "sd b": b.std(), "V apical": a.mean(), "sd a": a.std(), "difference": a.mean() - b.mean(),
                     "p": p, "means ok": within(b.mean(), e[0], e[1]) and within(a.mean(), e[2], e[3])})
print(pd.DataFrame(rows).round(4).to_string())
print("XE991 effect tests on V0.5 (pigmented, x-a; all groups with colony, x-b):")
print(effect_tests(df, "V05").round(4))
pig = c144[c144.pigmented & c144.region.notna()]
mn = pig.groupby(["region", "xe991"]).V05.mean().unstack()
print(mn.round(4))
sh = (mn[True] - mn[False])
print("h-a shift basal", round(sh["basal"], 4), "apical", round(sh["apical"], 4), "average", round(sh.mean(), 4), "V")
for y in ["QTm", "CLW", "alphaB"]:
    print("other parameter", y, ":")
    print(effect_tests(df, y).round(4))

**Result: reproduced.** Under r-a, V0.5 of pigmented XE991 cells is -0.0623 V at the base against -0.0358 V at the apex (ANOVA p = 0.0205), both within tolerance. The data set has 5 basal and 12 apical cells, where the paper has 4 and 11.

**Result: reproduced.** Under r-a, V0.5 of pigmented cells without XE991 is -0.0907 V basal (12 cells) against -0.0552 V apical (15 cells), with ANOVA p = 0.0005, which is below 0.001. Both means are within tolerance. Under r-b p = 0.0008.

**Result: reproduced.** Under r-a there are 11 basal and 30 apical albino cells, the paper's counts. V0.5 is -0.0710 against -0.0573 V (difference 0.0138 V), with ANOVA p = 0.0697, the paper's value; the test does not find the difference. Under r-b (3 and 15 cells) p = 0.0165.

**Result: reproduced.** The XE991 effect on V0.5 is significant, with p = 0.0019 (x-a) and 0.0007 (x-b). The shift is 0.0239 V by region-matched means (h-a: basal 0.0284, apical 0.0193 V) and 0.0225 V by the coefficient (h-b), both within 0.020 +/- 0.010 V. QTm, CLW and alphaB show no XE991 effect under x-a and x-b (p 0.27 to 0.86).

## v05-not-correlated-voltage-drop

> The increase of V0.5 is not correlated with the pipette voltage drop computed at V0.5 or at
> positive voltages, with or without XE991.

In [ ]:
rows = []
for cells, s in [("144 mM", vc.join(df[["V05", "chloride_mM"]]).query("chloride_mM == 144")), ("all 89", vc.join(df[["V05"]]))]:
    for x_ in [True, False]:
        g = s[s.xe991 == x_]
        for col in ["drop_V05", "drop_pos"]:
            pr = stats.pearsonr(g.V05, g[col])
            sp = stats.spearmanr(g.V05, g[col])
            rows.append({"cells": cells, "XE991": x_, "drop": col, "n": len(g), "pearson r": pr.statistic, "p": pr.pvalue,
                         "spearman rho": sp.statistic, "p spearman": sp.pvalue})
print(pd.DataFrame(rows).round(4))

**Result: reproduced (resting on the assumed test).** None of the four Pearson correlations of V0.5 with the fractional drop is significant (144 mM cells): with XE991, p = 0.97 at V0.5 and 0.088 at positive voltages; without, p = 0.10 and 0.065. In the Spearman variant, the XE991 drop at positive voltages correlates (rho 0.51, p = 0.038, 17 cells).

## fig3a-eq3a-fit and fig3a-xe991-fit

> Binned by lateral membrane capacitance (5 pF), V0.5 falls as the charge density dVmin_exp rises,
> as Eq. 3a predicts; the one-parameter fit gives Vd 0.033 V (0.034 V in the legend, adjusted R2
> 0.916), and an unconstrained linear fit has slope -0.401 (adjusted R2 0.95).

> Cells with XE991 follow Eq. 3a with Vd 0.05 V (0.0499 V, adjusted R2 0.793), and an
> unconstrained linear fit of slope -0.37 (adjusted R2 0.87).

Readings: bin edges at multiples of 5 pF (e-a) or from the smallest CLW (e-b); Eq. 3a R2 with (w-a)
or without (w-b) the weights; the unconstrained fit weighted (u-a) or not (u-b). For XE991, the 17
cells with 144 mM chloride and all 20 (all are stored as 60 uM, so reading k-b is k-a). Bins: y-1
(adopted, see the header) leaves out bins whose mean dVmin is not in (0, 0.4] V; y-0 keeps all bins;
y-2 leaves out cells with dVmin outside that range and bins the rest.

In [ ]:
def fig3a(s, col="CLW", width=5.0, lim="y-1"):
    """Eq. 3a and unconstrained fits on bins of `col`. lim y-1: bins with mean dVmin in (0, 0.4] V
    (ear1.in_range, the adopted reading); y-0: all bins; y-2: cells with dVmin in (0, 0.4] V, then binned."""
    if lim == "y-2":
        s = s[(s.dVmin > 0) & (s.dVmin <= ear1.DVMIN_MAX)]
    rows = []
    for e_, start in [("e-a", 0.0), ("e-b", s[col].min())]:
        b = ear1.bin_cells(s, col, width, start)
        if lim == "y-1":
            b = ear1.in_range(b)
        for w_ in ["w-a", "w-b"]:
            f = ear1.fit_eq3a(b.dVmin, b.V05, b.n, weighted_r2=(w_ == "w-a"))
            for u_ in ["u-a", "u-b"]:
                u = ear1.ols(b.dVmin, b.V05, b.n if u_ == "u-a" else None)
                rows.append({"limit": lim, "edges": e_, "R2": w_, "unconstrained": u_, "bins": len(b), "cells": int(b.n.sum()),
                             "Vd": f["Vd"], "adj R2 Eq3a": f["adj_r2"], "slope": u["slope"], "intercept": u["intercept"],
                             "adj R2 linear": u["adj_r2"]})
    return pd.DataFrame(rows)


def judge3a(t_, e):
    ok = t_.apply(lambda r: [within(r.Vd, e[0], e[1]), within(r["adj R2 Eq3a"], e[2], e[3]), within(r.slope, e[4], e[5]),
                             within(r["adj R2 linear"], e[6], e[7])], axis=1)
    t_ = t_.assign(ok=ok.apply(lambda v: "".join("y" if a else "n" for a in v)))
    return t_


E3A = (0.034, 0.0034, 0.916, 0.0916, -0.401, 0.0401, 0.95, 0.095)
E3X = (0.0499, 0.00499, 0.793, 0.0793, -0.37, 0.037, 0.87, 0.087)
LIMS = ["y-1", "y-0", "y-2"]
fits = {}
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, x_ = sets(d)
    for setn, s, e in [("no XE991", n_, E3A), ("XE991 144 mM", x_, E3X), ("XE991 all 20", d[d.xe991], E3X)]:
        fits[(lab, setn)] = pd.concat([judge3a(fig3a(s, lim=l), e) for l in LIMS], ignore_index=True)
        print((lab, setn), " (ok flags: Vd, Eq3a R2, slope, linear R2)")
        print(fits[(lab, setn)].round(4).to_string())
print("5 pF bins (e-a), all bins; y-1 leaves out bins with mean dVmin outside (0, 0.4] V:")
print(ear1.bin_cells(no, "CLW", 5)[["lo", "n", "CLW", "dVmin", "V05", "V05_sem"]].round(4))
print(ear1.bin_cells(xe, "CLW", 5)[["lo", "n", "CLW", "dVmin", "V05", "V05_sem"]].round(4))
print("XE991 cells of the first bin:", xe[xe.CLW < 5][["orig", "CL", "CSB", "CNLW", "CLW", "QTm", "dVmin", "V05"]].round(3).to_dict("index"))
print("bins left out by y-1 (L-b): no XE991", int((~ear1.bin_cells(no, "CLW", 5).index.isin(ear1.in_range(ear1.bin_cells(no, "CLW", 5)).index)).sum()),
      "; XE991", int((~ear1.bin_cells(xe, "CLW", 5).index.isin(ear1.in_range(ear1.bin_cells(xe, "CLW", 5)).index)).sum()))
b = ear1.bin_cells(no, "CLW", 5)
bx = ear1.bin_cells(xe, "CLW", 5)
print("plotted dVmin range", round(min(b.dVmin.min(), bx.dVmin.min()), 3), "to", round(max(b.dVmin.max(), bx.dVmin.max()), 3),
      "V; the x axis is cut at 0.4 V, so", int((bx.dVmin > 0.4).sum()), "XE991 bin lies outside it")
fig, ax = plt.subplots(figsize=(4, 3))
ax.errorbar(b.dVmin, b.V05, b.V05_sem, fmt="o", mfc="none", label="no XE991")
ax.errorbar(bx.dVmin, bx.V05, bx.V05_sem, fmt="o", mfc="none", color="C3", label="XE991")
ax.set_xlim(0, 0.4)
ax.set_xlabel("dVmin_exp (V)")
ax.set_ylabel("V0.5 (V)")
ax.legend()
plt.show()

**Result: reproduced (resting on the assumed CSB rule).** Under L-b, with bin edges from the smallest CLW (e-b), the Eq. 3a fit gives Vd 0.0311 V with adjusted R2 0.931 (w-a) or 0.949 (w-b), and the weighted unconstrained fit (u-a) has slope -0.4245 with adjusted R2 0.948: all four values are within tolerance. With edges at multiples of 5 pF (e-a), Vd and both R2 are within (0.0311 V, 0.942, 0.949), but the slope is -0.4412, outside -0.401 +/- 0.0401 by 0.0001; unweighted (u-b) the slopes are -0.477 and -0.473. The y-1 limit leaves out no bin without XE991, so y-0 gives the same values; y-2 drops one cell (dVmin above 0.4 V) and moves Vd to 0.0299 V, outside. Under L-c, Vd is 0.0215 V and the slope -0.59 to -0.63, both outside, under every reading.

**Result: reproduced (resting on the assumed CSB rule and the assumed dVmin range).** Under L-b and y-1, the bin of the two smallest XE991 cells (mean dVmin 0.80 V; file 57 has CLW 0.48 pF and dVmin 1.33 V) is left out, and the other 4 bins (15 cells with 144 mM chloride) give Vd 0.0505 V with adjusted R2 0.847, and a weighted unconstrained slope of -0.377 with adjusted R2 0.921 (e-a, w-a, u-a): all four within tolerance. With all 20 cells, edges e-b give 0.0476 V, 0.745, -0.355 and 0.788, also all within. Binning only the cells within the range (y-2) also passes under e-b (0.0522 V, 0.715, -0.336, 0.911). Keeping the bin (y-0), the outcome before this revision, gives Vd 0.0847 V, an Eq. 3a R2 of -43 and a slope of -0.038, so no value passes. Under L-c with y-1, Vd is 0.0432 V, outside.

## fig3a-high-density-deviation

> Below a charge density of 0.25 V the data lie close to the Eq. 3a line, while above it V0.5 is
> larger than expected and more variable.

In [ ]:
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, x_ = sets(d)
    vd = ear1.fit_eq3a(ear1.bin_cells(n_, "CLW", 5).dVmin, ear1.bin_cells(n_, "CLW", 5).V05, ear1.bin_cells(n_, "CLW", 5).n)["Vd"]
    res = n_.V05 - (vd - n_.dVmin / 2)
    hi, lo = res[n_.dVmin > 0.25], res[n_.dVmin <= 0.25]
    print(lab, "Vd", round(vd, 4), "; cells above 0.25 V:", len(hi), "below:", len(lo))
    if len(hi) >= 2:
        tt = stats.ttest_1samp(hi, 0)
        F = hi.var() / lo.var()
        pF = stats.f.sf(F, len(hi) - 1, len(lo) - 1)
        print("  per cell: mean residual above", round(hi.mean(), 4), "V (t-test p", round(tt.pvalue, 4), "), below", round(lo.mean(), 4),
              "; variance ratio", round(F, 3), "p", round(pF, 4))
    else:
        print("  per cell: too few cells above 0.25 V for the tests; mean residual above", round(hi.mean(), 4) if len(hi) else None)
    bs = ear1.bin_cells(n_, "dVmin", 0.01)
    rb = bs.V05 - (vd - bs.dVmin / 2)
    hb, lb = rb[bs.dVmin > 0.25], rb[bs.dVmin <= 0.25]
    print("  0.01 V bins: above", len(hb), "bins, mean residual", round(hb.mean(), 4) if len(hb) else None, "; below", len(lb),
          "bins, mean", round(lb.mean(), 4), "; sd above", round(hb.std(), 4) if len(hb) > 1 else None, "below", round(lb.std(), 4))
    if len(hb) >= 2:
        Fb = hb.var() / lb.var()
        print("  0.01 V bins: t-test of the mean residual above, p", round(stats.ttest_1samp(hb, 0).pvalue, 4),
              "; variance ratio", round(Fb, 3), "p", round(stats.f.sf(Fb, len(hb) - 1, len(lb) - 1), 4))
print("dVmin range (no XE991, L-b):", round(no.dVmin.min(), 3), "to", round(no.dVmin.max(), 3), "V")

**Result: reproduced (resting on the assumed CSB rule and on tests the paper does not name).** Under L-b, 8 cells without XE991 have dVmin above 0.25 V and 61 below. On the 0.01 V bins of Fig. S4, the reading the criterion names for the binned data, the 5 bins above 0.25 V have mean residual 0.0351 V (t-test p = 0.042), larger in magnitude than the -0.0032 V of the 14 bins below, and their residuals spread more (SD 0.0267 against 0.0130 V, F-test p = 0.021): all three conditions hold. Per cell, the mean residual above is 0.0284 V (p = 0.012) against -0.0037 V below, but the variance ratio is 1.15 (p = 0.34), so that reading gives only the partial tier. Under L-c only 2 cells lie above 0.25 V (residual 0.0227 V, p = 0.41).

## figs4-fine-bins

> With cells binned by charge density in 0.01 V bins (dVmin_exp < 0.4 V fitted), Eq. 3a gives Vd
> 0.030 V (adjusted R2 0.389) without XE991 and Vd 0.054 V (adjusted R2 0.737) with XE991.

In [ ]:
rows = []
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, x_ = sets(d)
    for setn, s, e in [("no XE991", n_, (0.030, 0.003, 0.389, 0.0389)), ("XE991", x_, (0.054, 0.0054, 0.737, 0.0737))]:
        b = ear1.bin_cells(s, "dVmin", 0.01)
        for lim in ([True] if setn == "no XE991" else [True, False]):
            bb = b[b.dVmin < 0.4] if lim else b
            for w_ in ["w-1", "w-2"]:
                f = ear1.fit_eq3a(bb.dVmin, bb.V05, bb.n if w_ == "w-1" else None)
                rows.append({"CLW": lab, "set": setn, "0.4 V limit": lim, "weights": w_, "bins": len(bb), "Vd": f["Vd"], "adj R2": f["adj_r2"],
                             "Vd ok": within(f["Vd"], e[0], e[1]), "R2 ok": within(f["adj_r2"], e[2], e[3])})
print(pd.DataFrame(rows).round(4).to_string())

**Result: partially reproduced (resting on the assumed CSB rule).** Under L-b, without XE991 the weighted fit (w-1) gives Vd 0.0299 V with adjusted R2 0.396, both within tolerance (0.0338 V and R2 0.503 unweighted). With XE991 and the 0.4 V limit it gives Vd 0.0522 V (w-1) or 0.0556 V (w-2), within 0.054 +/- 0.0054, but R2 0.605 or 0.625, outside 0.737 +/- 0.074. Without the limit the XE991 fit fails (Vd 0.0847 V, R2 -52). Both Vd are within tolerance and one R2 is not, which is the partial tier. Under L-c no reading puts both Vd within (0.0215 V without XE991).

## tables3-by-region

> Similar trends of V0.5 with charge density are found when the charge density is computed per
> cell and averaged by cochlear region.

In [ ]:
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, x_ = sets(d)
    for reg in ["region", "region_rb"]:
        allok = True
        for g, s in {"pig XE991": x_[x_.pigmented], "pig no XE991": n_[n_.pigmented], "albino": n_[~n_.pigmented]}.items():
            t_ = s.groupby(reg).agg(n=("V05", "size"), dVmin=("dVmin", "mean"), V05=("V05", "mean"))
            hi_ = t_.dVmin.idxmax()
            ok_ = t_.loc[hi_, "V05"] == t_.V05.min()
            allok &= bool(ok_)
            print(lab, reg, g, t_.round(4).to_dict("index"), "higher dVmin has more negative V0.5:", bool(ok_))
        print("  all groups:", allok)

**Result: reproduced (resting on the assumed CSB rule).** In each of the three groups, under both region readings, the region with the higher mean dVmin (basal) has the more negative mean V0.5. Under L-b, pigmented cells without XE991 give 0.2465 V and -0.0907 V basal against 0.1750 V and -0.0552 V apical. L-c gives the same ordering in every group. Table S3's values are not available, so no number is compared.

## capacitance-asymmetry

> Linear capacitance at hyperpolarized voltages exceeds that at positive voltages by 2.76 +/- 1.32
> pF (62 cells); excluding differences above 5 pF, the difference is 2.87 pF for apical and 2.53 pF
> for basal cells, a 13% and 21% increase in CLW.

CLn is not stored, and the paper does not say how it was taken, only that it could not be
estimated where "the capacitance had not reached its plateau (hyperpolarized) value". Readings of
CLn: the mean C of the two most negative ramp points (`first2`, which gives the Fig. S1 legend's
25.94 pF for its cell) and the mean C at or below -0.15 V. Plateau rule: the slope of C over the
points at or below -0.15 V is less than a fraction of the steepest 5-point slope of the
hyperpolarized flank; the fraction is not given, so it is scanned. Assumption: the fraction whose
count is nearest the paper's 62 cells is adopted (the paper's count is the one fact about its
rule); every fraction is shown. The C of the first eight ramp points is printed to show whether the
two used by `first2` carry a settling transient.

In [ ]:
cl = pd.DataFrame({f: ear1.cln_readings(tr[f]["V"], tr[f]["C"], df.loc[f, "CL"]) for f in df.index}).T
cl = cl.join(df[["CL", "CLW", "region", "xe991", "chloride_mM", "area_um2", "QTm", "V05", "z"]])
cl["ratio"] = cl.plateau_slope / cl.rise_slope
print("cell 99:", cl.loc[f99, ["first2", "below_m015"]].round(3).to_dict())


def asym(cl, rd, th, clw="CLW"):
    s = cl[cl.ratio < th]
    dC = s[rd] - s.CL
    k = s[dC <= 5]
    dk = k[rd] - k.CL
    out = {"reading": rd, "fraction": th, "n": len(s), "mean dC": dC.mean(), "sd dC": dC.std(), "n dC<=5": len(k)}
    for reg in ["apical", "basal"]:
        g = k[k.region == reg]
        dd = g[rd] - g.CL
        out[f"n {reg}"] = len(g)
        out[f"dC {reg}"] = dd.mean()
        out[f"m-a {reg}"] = (dd / g[clw]).mean()
        out[f"m-b {reg}"] = dd.mean() / g[clw].mean()
    return out


scan = pd.DataFrame([asym(cl, rd, th) for rd in ["first2", "below_m015"] for th in [0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 1.0, np.inf]])
print(scan.round(3).to_string())
TH = scan[scan.reading == "first2"].assign(gap=lambda x: (x.n - 62).abs()).sort_values(["gap", "fraction"]).fraction.iloc[0]
main = asym(cl, "first2", TH)
print("adopted fraction", TH, ":", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in main.items()})
E_AS = [(main["mean dC"], 2.76, 1.32), (main["dC apical"], 2.87, 1.27), (main["dC basal"], 2.53, 1.47),
        (main["m-a apical"], 0.13, 0.013), (main["m-a basal"], 0.21, 0.021)]
print("m-a within tolerance:", [within(a, b, c) for a, b, c in E_AS], " count within 62 +/- 3:", within(main["n"], 62, 3))
print("m-b apical", round(main["m-b apical"], 3), "ok", within(main["m-b apical"], 0.13, 0.013), "; m-b basal", round(main["m-b basal"], 3),
      "ok", within(main["m-b basal"], 0.21, 0.021))
clB = cl.copy()
clB["CLW"] = clB.CL - ear1.CNLW_C
mB = asym(clB, "first2", TH)
print("L-c 4.82: m-a apical", round(mB["m-a apical"], 3), "basal", round(mB["m-a basal"], 3), "; m-b apical", round(mB["m-b apical"], 3),
      "basal", round(mB["m-b basal"], 3))
# Do the first two ramp points carry a settling transient? C minus CL at the first eight ramp points
# (-0.18 V upward, about 0.0031 V apart) for the first six cells and the median over all cells.
pts = {}
for f in df.index:
    i0, i1 = ear1.ramp_indices(tr[f]["V"])
    pts[f] = tr[f]["C"][i0:i0 + 8] - df.loc[f, "CL"]
pts = pd.DataFrame(pts, index=[f"point {k}" for k in range(8)]).T
print(pts.head(6).round(2).to_string())
print("median over 89 cells:", pts.median().round(3).to_dict())
d01 = pts.iloc[:, :2].mean(axis=1) - pts.iloc[:, 2:].mean(axis=1)
print("mean of points 0-1 minus mean of points 2-7 (pF): median", round(d01.median(), 3), "; quartiles",
      round(d01.quantile(0.25), 3), "to", round(d01.quantile(0.75), 3), "; cells where the first two are higher:", int((d01 > 0).sum()), "of", len(d01))

**Result: not reproduced (resting on the assumed plateau rule and CSB rule).** The adopted plateau fraction of 0.4 keeps 62 cells. Their mean dC is 2.89 pF (SD 2.42, against 1.32). After dropping dC > 5 pF (53 cells), dC is 2.40 pF apical (35 cells) and 1.88 pF basal (18 cells), all within tolerance. As a share of CLW (L-b) the apical value is 15.4% by m-a, outside 13 +/- 1.3%, and 14.1% by m-b, within. The basal share is 36.1% (m-a) or 29.9% (m-b), outside 21 +/- 2.1%. Under L-c the shares are 14.2% and 28.6% (m-a). CLn at or below -0.15 V gives larger dC (3.68 pF for 62 cells). The first two ramp points show no settling transient that would raise them: over all 89 cells the median C - CL is 3.48 and 3.44 pF at the first two points and rises to 5.02 pF by the eighth, and the first two lie below the next six in 80 of 89 cells (median by 0.88 pF), as expected on the rising flank toward the peak. The three differences are within tolerance, but the criterion needs every value within tolerance and has no partial tier, and both shares of CLW fail for basal cells.

## figs5-specific-capacitance

> The specific capacitance of the lateral membrane is 0.0095 pF/um2 at positive voltages
> (intercept 2.14 pF, adjusted R2 0.79, 83 cells) and 0.01106 pF/um2 at negative voltages (fit
> with the same intercept, adjusted R2 0.807, 59 cells).

Area readings: (a-a) the stored area; (a-b) 2 pi r (l - r) and (a-c) 2 pi r l, with l the lateral
wall length (83 cells; the cell length is stored for 35 only). CLn as in capacitance-asymmetry
(adopted plateau fraction, `first2`).

In [ ]:
r_ = df.diameter_um / 2
areas = {"a-a stored": df.area_um2, "a-b 2 pi r (l - r)": 2 * np.pi * r_ * (df.lw_length_um - r_), "a-c 2 pi r l": 2 * np.pi * r_ * df.lw_length_um}
cln_ok = cl[cl.ratio < TH]
rows = []
for an, A in areas.items():
    g = df[A.notna()].assign(A=A)
    p_ = ear1.ols(g.A, g.CL)
    h = cln_ok[["first2"]].join(g[["A"]], how="inner").dropna()
    yk = h.first2 - p_["intercept"]
    k = np.sum(h.A * yk) / np.sum(h.A**2)
    sse = np.sum((yk - k * h.A) ** 2)
    adjn = 1 - (sse / (len(h) - 1)) / (np.sum((h.first2 - h.first2.mean()) ** 2) / (len(h) - 1))
    rows.append({"area": an, "n": p_["n"], "slope pF/um2": p_["slope"], "intercept pF": p_["intercept"], "adj R2": p_["adj_r2"],
                 "n CLn": len(h), "slope CLn": k, "adj R2 CLn": adjn,
                 "ok": "".join("y" if v else "n" for v in [within(p_["slope"], 0.0095, 0.00095), within(p_["intercept"], 2.14, 0.214),
                                                            within(p_["adj_r2"], 0.79, 0.079), within(k, 0.011062, 0.0016), within(adjn, 0.807, 0.0807)])})
print(pd.DataFrame(rows).round(5).to_string())
print("(ok flags: slope, intercept, R2, CLn slope, CLn R2)")
print("cells with CLn under other plateau fractions and an area:", {th: int(((cl.ratio < th) & df.area_um2.notna()).sum()) for th in [0.2, 0.3, 0.4, 0.5, np.inf]})

**Result: reproduced (resting on the assumed CLn rule for the negative-voltage fit).** On the stored area, CL fits with slope 0.00952 pF/um2, intercept 2.25 pF and adjusted R2 0.794 (83 cells). CLn with the same intercept gives slope 0.01093 pF/um2 and adjusted R2 0.787 (57 cells, where the paper has 59). All five values are within tolerance. The other two area formulas move the intercept outside tolerance.

## fig3b-negative-clw-fit

> With CLW taken at negative potentials (44 cells, no XE991), Eq. 3a fits with an intercept of 0.02
> V (adjusted R2 0.91), while an unconstrained linear fit has slope -0.664 and intercept 0.046 V
> (adjusted R2 0.97).

CLWn = CLW + dC for the cells without XE991 that have a CLn (adopted plateau fraction, `first2`);
dVmin^n = QTm / CLWn; binned by CLWn in 5 pF bins as Fig. 3a; cells with dC > 5 pF kept or dropped; CLn by `first2` and at or below -0.15 V; bins by y-1 (adopted)
and y-0. The bins and the dC of every cell are printed.

In [ ]:
E3B = (0.02, 0.002, 0.91, 0.091, -0.664, 0.0664, 0.97, 0.097)
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, _ = sets(d)
    h0 = n_.join(cl[["first2", "below_m015", "ratio"]])
    h0 = h0[h0.ratio < TH]
    for rd in ["first2", "below_m015"]:
        h = h0.assign(dC=h0[rd] - h0.CL)
        for keep in ["kept", "dropped"]:
            s = h if keep == "kept" else h[h.dC <= 5]
            s = s.assign(CLW=s.CLW + s.dC)
            s = s.assign(dVmin=s.QTm / s.CLW)
            print(lab, "CLn", rd, "| dC > 5 pF", keep, ": cells", len(s), "; dC < 0:", int((s.dC < 0).sum()), "; CLWn < 0:", int((s.CLW < 0).sum()))
            print(ear1.bin_cells(s, "CLW", 5)[["lo", "n", "CLW", "dVmin", "V05"]].round(4).to_string())
            t_ = pd.concat([judge3a(fig3a(s, lim=l), E3B) for l in LIMS], ignore_index=True)
            t_["intercept ok"] = [within(a, 0.046, 0.0046) for a in t_.intercept]
            print(t_.round(4).to_string())
h = no.join(cl[["first2", "ratio"]])
h = h[h.ratio < TH]
dC = (h.first2 - h.CL).sort_values()
print("dC per cell (first2, L-b cells without XE991):", dC.round(2).tolist())
print("cells with dC < 0:", h.loc[dC[dC < 0].index, ["orig", "CL", "CLW", "first2"]].round(3).to_dict("index"))

**Result: not reproduced.** Under L-b, `first2` and dC > 5 pF kept, 46 cells qualify (the legend counts 44). Four have dC < 0, and one (file 18, dC -5.0 pF) has a negative CLWn of -0.95 pF, which forms a bin of dVmin -0.70 V. Keeping that bin (y-0), the outcome before this revision, the unconstrained slope is +0.030. Under y-1 it is left out, and the remaining 7 bins (45 cells) give Vd 0.0172 V, outside 0.02 +/- 0.002, an Eq. 3a R2 of 0.33, and an unconstrained slope of -0.403 with intercept 0.0027 V and R2 0.22. Dropping dC > 5 pF gives Vd 0.0233 V and slope -0.523 (e-a) or -0.703 (e-b). CLn at or below -0.15 V gives Vd 0.0133 V (kept) or 0.0214 V (dropped), with slopes from -0.27 to -0.97 and adjusted R2 at most 0.74. No reading puts Vd and the slope within tolerance together, and no R2 comes near 0.91 or 0.97. Under L-c, Vd is 0.0117 V (y-1).

## mean-slope-half

> The mean of the three unconstrained slopes for guinea pig cells (-0.4, -0.37, -0.66) is about
> -0.48, close to the -1/2 Eq. 3a predicts for more than one hyperpolarized state.

The slopes are the first reading of each fit (edges e-a, weighted unconstrained fit u-a; for
XE991 the 144 mM cells; for Fig. 3b `first2` with dC > 5 pF kept), bins under y-1, with the
unweighted fits and y-0 as variants. Every combination of the readings is also shown.

In [ ]:
import itertools


def slope3(d, e_, u_, lim, rd="first2", keep="kept"):
    c, n_, x_ = sets(d)
    q = "edges == @e_ and unconstrained == @u_"
    s1 = fig3a(n_, lim=lim).query(q).slope.iloc[0]
    s2 = fig3a(x_, lim=lim).query(q).slope.iloc[0]
    h = n_.join(cl[["first2", "below_m015", "ratio"]])
    h = h[h.ratio < TH]
    h = h.assign(dC=h[rd] - h.CL)
    if keep == "dropped":
        h = h[h.dC <= 5]
    h = h.assign(CLW=h.CLW + h.dC)
    h = h.assign(dVmin=h.QTm / h.CLW)
    s3 = fig3a(h, lim=lim).query(q).slope.iloc[0]
    return s1, s2, s3, np.mean([s1, s2, s3])


for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    for lim in ["y-1", "y-0"]:
        for u_ in ["u-a", "u-b"]:
            s = slope3(d, "e-a", u_, lim)
            print(lab, lim, "e-a", u_, "slopes", np.round(s[:3], 3), "mean", round(s[3], 3), "within -0.48 +/- 0.048:", within(s[3], -0.48, 0.048))
# Every combination of the readings of the three fits (edges, weights, CLn reading, dC > 5 pF), y-1.
allm = pd.DataFrame([dict(edges=e_, unc=u_, cln=rd, keep=k, mean=slope3(df, e_, u_, "y-1", rd, k)[3])
                     for e_, u_, rd, k in itertools.product(["e-a", "e-b"], ["u-a", "u-b"], ["first2", "below_m015"], ["kept", "dropped"])])
allm["ok"] = [within(m, -0.48, 0.048) for m in allm["mean"]]
print("L-b, y-1, all 16 reading combinations: mean slope", round(allm["mean"].min(), 3), "to", round(allm["mean"].max(), 3),
      "; within tolerance in", int(allm.ok.sum()), "of 16")
print(allm.round(3).to_string())

**Result: not reproduced (resting on the assumed CSB rule, dVmin range and choice of readings).** Under L-b and y-1, the three slopes are -0.441 (fig3a-eq3a-fit), -0.377 (XE991) and -0.403 (Fig. 3b), with mean -0.407, outside -0.48 +/- 0.048; the unweighted fits give -0.328. The readings change the outcome: over the 16 combinations of edges, weighting, CLn reading and the dC > 5 pF rule, the mean runs from -0.598 to -0.328 and is within tolerance in 7. The first reading of each fit is kept, as before. With the slopes of the readings recorded for each claim (e-b for fig3a-eq3a-fit, -0.4245) the mean is -0.402. Keeping all bins (y-0), the outcome before this revision, gives -0.150. Under L-c with y-1 the mean is -0.515, within tolerance.

## fig4a-vh-rises-vd-constant

> The hyperpolarized pseudo-minimum Vh increases monotonically by threefold across cells while the
> depolarized pseudo-minimum Vd stays about constant, evidence for more than one hyperpolarized
> state.

In [ ]:
def fig4a(d, lim):
    c, n_, x_ = sets(d)
    out = []
    for o_, parts in [("o-a no XE991", [n_]), ("o-b pooled", [n_, x_])]:
        bs = [ear1.bin_cells(p, "CLW", 5) for p in parts]
        b = pd.concat([ear1.in_range(x) if lim == "y-1" else x for x in bs], ignore_index=True)
        b["Vh"] = b.V05 - b.dVmin / 2
        b["Vd"] = b.V05 + b.dVmin / 2
        inv_clw = int((np.diff(b.sort_values("CLW").Vh.to_numpy()) < 0).sum())
        b = b.sort_values("V05")
        rho = stats.spearmanr(b.Vh, b.V05).statistic
        inv = int((np.diff(b.Vh.to_numpy()) < 0).sum())
        ratio = b.Vh.abs().max() / b.Vh.abs().min()
        qa = (b.Vd.max() - b.Vd.min()) < (b.Vh.max() - b.Vh.min()) / 3
        r = ear1.ols(b.V05, b.Vd, b.n)
        out.append({"limit": lim, "reading": o_, "bins": len(b), "spearman Vh~V0.5": rho, "inversions": inv, "inversions by CLW": inv_clw,
                    "|Vh| max/min": ratio, "Vh range": b.Vh.max() - b.Vh.min(), "Vd range": b.Vd.max() - b.Vd.min(), "q-a": qa,
                    "q-b slope": r["slope"], "q-b p": r["p_slope"], "ratio ok": within(ratio, 3, 0.3)})
        print(lim, o_, b[["n", "CLW", "V05", "dVmin", "Vh", "Vd"]].round(4).to_dict("list"))
    return pd.DataFrame(out)


print("L-b")
print(pd.concat([fig4a(df, "y-1"), fig4a(df, "y-0")]).round(4).to_string())
print("L-c 4.82")
print(pd.concat([fig4a(dfB, "y-1"), fig4a(dfB, "y-0")]).round(4).to_string())

**Result: not reproduced.** Under L-b, in the 5 pF bins without XE991 (o-a), Vh rises monotonically with V0.5 (Spearman 1.0, no inversions), but the ratio of largest to smallest |Vh| is 2.67, below the bound of 2.7. Vd varies over 0.010 V against 0.142 V for Vh, so Vd is constant under q-a, and under q-b its slope on V0.5 (-0.087) does not differ from zero (p = 0.48). Pooling the XE991 bins (o-b) under y-1 gives a ratio of 2.76, within tolerance, with Vd constant (q-a; q-b p = 0.47), but Vh has 2 inversions when ordered by V0.5 and 3 when ordered by CLW (Spearman 0.976), more than the one allowed. With all bins (y-0) the pooled ratio is 5.61 with three inversions. No reading meets the criterion, which has no partial tier. Under L-c the o-a ratio is 2.60.

## fig4d-cleq-two-limits

> The capacitance of the membrane where one sensor resides, C_Leq = n e0 / dVmin_exp, has two
> limits, 0.584 (+/- 0.065) aF at mean V0.5 below -0.06 V, rising almost exactly twofold to 1.07
> (+/- 0.038) aF as V0.5 rises to -0.03 V.

Groups (j-a) 5 pF CLW bins, (j-b) 0.01 V dVmin bins, (j-c) cells; cells with and without XE991
with 144 mM chloride. Lower limit: mean C_Leq over groups with V0.5 <= -0.056 V; upper: groups
with V0.5 >= -0.043 V (Table 3 ranges); the text's split at -0.06 V is a variant.
Groups under y-1 (adopted, see the header); y-0 keeps every group.

In [ ]:
def limits(d, how, lo_cut=-0.056, hi_cut=-0.043, lim="y-1"):
    c, _, _ = sets(d)
    g = ear1.groups(c, how, limit=(lim == "y-1"))
    lo, hi = g[g.V05 <= lo_cut], g[g.V05 >= hi_cut]
    sp = stats.spearmanr(g.C_Leq_aF, g.V05)
    return g, lo, hi, {"limit": lim, "grouping": how, "split": f"{lo_cut}/{hi_cut}", "groups": len(g), "n low": len(lo),
                       "C_Leq low": lo.C_Leq_aF.mean(), "n high": len(hi), "C_Leq high": hi.C_Leq_aF.mean(),
                       "between": len(g) - len(lo) - len(hi), "high/low": hi.C_Leq_aF.mean() / lo.C_Leq_aF.mean(),
                       "spearman": sp.statistic, "p": sp.pvalue,
                       "low ok": within(lo.C_Leq_aF.mean(), 0.584, 0.065), "high ok": within(hi.C_Leq_aF.mean(), 1.07, 0.038)}


for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    rows = [limits(d, how, *cuts, lim=l)[3] for l in ["y-1", "y-0"] for cuts in [(-0.056, -0.043), (-0.06, -0.06)]
            for how in ["clw5", "dv001", "cell"]]
    print(lab)
    print(pd.DataFrame(rows).round(4).to_string())
for l in ["y-1", "y-0"]:
    print("5 pF CLW groups (L-b,", l, "):")
    print(limits(df, "clw5", lim=l)[0].round(4).to_string())
print("ranges: C_Leq per cell", round(c144.C_Leq_aF.min(), 3), "to", round(c144.C_Leq_aF.max(), 3), "aF")
print("file 57:", df.loc[57, ["CL", "CSB", "CNLW", "CLW", "QTm", "dVmin"]].round(3).to_dict())

**Result: not reproduced (resting on the assumed CSB rule and the assumed dVmin range).** Under L-b and y-1, the 5 pF groups (j-a) give a lower limit of 0.6496 aF (4 groups), outside 0.584 +/- 0.065 by 0.0006 aF, and an upper limit of 1.048 aF (3 groups), within 1.07 +/- 0.038; C_Leq rises with V0.5 (Spearman 0.92, p = 0.0002). The 0.01 V groups (j-b) give 0.625 aF, within, and 0.979 aF, outside; per cell (j-c) 0.704 and 0.894 aF, both outside. Splitting at -0.06 V gives 0.650 and 0.923 aF (j-a). No grouping puts both limits within tolerance. Keeping every group (y-0), the outcome before this revision, the j-a lower limit is 0.560 aF and the claim passes; that value rests on the one XE991 group of the two smallest cells, with mean dVmin 0.80 V and C_Leq 0.20 aF, which y-1 leaves out as for the Eq. 3a fits. The upper limit of 1.048 aF includes a group of one cell (1.29 aF). Under L-c, j-a gives 0.748 and 1.080 aF.

## fig1-two-state-bounds

> The charge predicted by Eq. 6b for a membrane holding only one of the two sensor states (lambda
> 3.95 kB T with C_Leq 0.584 aF, and lambda 2.16 kB T with C_Leq 1.07 aF) bounds the measured
> charges, cells between the two limits holding mixtures of the two states.

In [ ]:
lam = np.array([3.95, 2.16]) * ear1.KT
cleq = np.array([0.584, 1.07]) * 1e-18
lines = np.sqrt(2 * lam / cleq)
print("dVmin of the two lines:", np.round(lines, 4), "V")
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, _, _ = sets(d)
    band = (c.dVmin >= 0.9 * lines[1]) & (c.dVmin <= 1.1 * lines[0])
    m1, m2 = c[c.CLW < 10].dVmin.median(), c[c.CLW >= 10].dVmin.median()
    print(lab, ": b-a fraction inside the widened band", round(band.mean(), 3), f"({int(band.sum())} of {len(c)})",
          "; above", int((c.dVmin > 1.1 * lines[0]).sum()), "below", int((c.dVmin < 0.9 * lines[1]).sum()),
          "; b-b median dVmin CLW<10", round(m1, 4), "ok", within(m1, lines[0], 0.1 * lines[0]), ", CLW>=10", round(m2, 4),
          "ok", within(m2, lines[1], 0.1 * lines[1]))

**Result: not reproduced (resting on the assumed CSB rule).** The two Eq. 6b lines are dVmin 0.2356 and 0.1287 V. Under L-b, 74 of the 86 cells (86.0%) lie between them widened by 10% (8 above, 4 below), short of 90% (reading b-a). Under b-b, the median dVmin is 0.2362 V for cells with CLW < 10 pF, within 10% of the upper line, but 0.1673 V for the others, not within 10% of 0.1287 V. Neither reading holds. Under L-c, 79 of 86 (91.9%) lie in the band.

## fig4c-sensor-area, fig4e-sensor-charge, table3-pathway-energies, alternative-limit

> The limiting membrane area of one wild type sensor is 51 to 121 nm2, with |V0.5| =
> 38.5 Amean^-1.15 (exponent -1.5 in the text).

> The charge moved per sensor is similar for all sensors, 0.873 +/- 0.0577 e0 at the 0.584 aF limit
> and 0.872 +/- 0.083 e0 at the 1.07 aF limit, 0.859 +/- 0.066 e0 over all wild type sensors.

> Pathway 1 has C_Leq 0.584 aF, V0.5 -0.076 V and z 0.859, lambda 3.97 kB T and barriers of 2.68
> (forward) and 0.125 kB T (reverse); pathway 2 has C_Leq 1.07 aF, V0.5 -0.031 V, lambda 2.16
> kB T and barriers of 1.19 (forward) and 0.145 kB T (reverse).

> Under the alternative limit of constant C_Leq (1.098 aF), 1.7 charges (1.44 to 1.92) would move
> a fraction delta = 0.535 (0.432 to 0.63) of the field.

All on the groups of fig4d-cleq-two-limits.

In [ ]:
def energies(z_, cleq_aF, v):
    q = z_ * ear1.E0
    lam_ = q**2 / (2 * cleq_aF * 1e-18) / ear1.KT
    qv = q * v / ear1.KT
    return lam_, (lam_ - qv) ** 2 / (4 * lam_), (lam_ + qv) ** 2 / (4 * lam_)


print("Eq. 6a and 7 as rebuilt, on Table 3's own inputs: pathway 1", np.round(energies(0.859, 0.584, -0.076), 3),
      "; pathway 2", np.round(energies(0.859, 1.07, -0.031), 3))
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    rows = []
    for lim in ["y-1", "y-0"]:
        for how in ["clw5", "dv001", "cell"]:
            g, lo, hi, _ = limits(d, how, lim=lim)
            nog = g[~g.xe991]
            ok_ = nog[(nog.V05 < 0) & (nog.A_nm2 > 0)]
            pw = {}
            for units, k in [("V", 1.0), ("mV", 1000.0)]:
                r = ear1.ols(np.log(ok_.A_nm2), np.log(-ok_.V05 * k))
                pw[units] = (np.exp(r["intercept"]), r["slope"])
            zl, zh, za = lo.z.mean(), hi.z.mean(), g.z.mean()
            wt = stats.ttest_ind(lo.z, hi.z, equal_var=False) if len(lo) > 1 and len(hi) > 1 else None
            e1 = energies(za, lo.C_Leq_aF.mean(), lo.V05.mean())
            e2 = energies(za, hi.C_Leq_aF.mean(), hi.V05.mean())
            n_alt = g.dVmin * 1.098e-18 / ear1.E0
            delta = g.z / n_alt
            rate = np.exp(-e2[1]) / np.exp(-2 * e1[1])
            rows.append({"reading": f"{lim} {how}", "rate ratio p2/p1 (dimer)": rate, "A min nm2": g.A_nm2.min(), "A max nm2": g.A_nm2.max(),
                         "A in 45.9-133.1": bool(g.A_nm2.min() >= 45.9 and g.A_nm2.max() <= 133.1),
                         "coef (V)": pw["V"][0], "exp (V)": pw["V"][1], "coef (mV)": pw["mV"][0], "exp (mV)": pw["mV"][1],
                         "z low": zl, "z high": zh, "z all": za, "Welch p": wt.pvalue if wt else np.nan,
                         "V0.5 p1": lo.V05.mean(), "V0.5 p2": hi.V05.mean(), "lambda p1": e1[0], "fwd p1": e1[1], "rev p1": e1[2],
                         "lambda p2": e2[0], "fwd p2": e2[1], "rev p2": e2[2],
                         "n alt mean": n_alt.mean(), "n alt min": n_alt.min(), "n alt max": n_alt.max(),
                         "delta mean": delta.mean(), "delta min": delta.min(), "delta max": delta.max()})
    t3 = pd.DataFrame(rows).set_index("reading").T
    print(lab)
    print(t3.round(4).to_string())
    for how in t3.columns:
        x = t3[how]
        print(" ", how, "| fig4c range ok", x["A in 45.9-133.1"], "| fig4e z low ok", within(x["z low"], 0.873, 0.0577), "z high ok",
              within(x["z high"], 0.872, 0.083), "z all ok", within(x["z all"], 0.859, 0.066),
              "| table3 V p1", within(x["V0.5 p1"], -0.076, 0.014), "V p2", within(x["V0.5 p2"], -0.031, 0.007),
              "lam p1", within(x["lambda p1"], 3.97, 0.397), "fwd p1", within(x["fwd p1"], 2.68, 0.268),
              "lam p2", within(x["lambda p2"], 2.16, 0.216), "fwd p2", within(x["fwd p2"], 1.19, 0.119),
              "| alt n", within(x["n alt mean"], 1.7, 0.17), "delta", within(x["delta mean"], 0.535, 0.0535),
              "n range in 1.296-2.112", bool(x["n alt min"] >= 1.296 and x["n alt max"] <= 2.112),
              "delta range in 0.389-0.693", bool(x["delta min"] >= 0.3888 and x["delta max"] <= 0.693))

**Result: not reproduced.** Under L-b and y-1, the sensor areas of the 5 pF groups run from 63.8 to 135.6 nm2; the upper end is beyond 51 to 121 nm2 widened by 10% (45.9 to 133.1 nm2). The other groupings run from 42.0 to 143.5 nm2 (j-b) and 42.0 to 152.6 nm2 (j-c). With all groups (y-0) the 5 pF groups start at 21.3 nm2 (the XE991 group of the two smallest cells). The power law over groups without XE991 has exponent -1.60 (j-a), -1.09 (j-b) or -0.68 (j-c), with coefficients 64.8, 6.92 and 1.15 in V and nm2. The j-a exponent is within 10% of the text's -1.5, but its coefficient is not 38.5. Under L-c with y-1 the 5 pF groups run from 72.2 to 132.1 nm2, within, but the exponent is -2.10.

**Result: reproduced (resting on the assumed CSB rule).** Under L-b and y-1, with the 5 pF groups (j-a), mean z is 0.913 at the lower limit, 0.883 at the upper and 0.875 over all groups, all within tolerance, and the two limits do not differ (Welch p = 0.34). The 0.01 V groups (j-b) give 0.911, 0.814 and 0.872 (p = 0.17), and per cell (j-c) 0.872 and 0.823 (p = 0.097), also within. With all groups (y-0), j-a puts z at the lower limit at 0.9325, outside 0.873 +/- 0.0577 by 0.002, while j-b and j-c pass. Under L-c, j-a gives 0.905 and 0.857.

**Result: reproduced (resting on the assumed CSB rule and the assumed dVmin range).** The rebuilt Eqs. 6a and 7 give Table 3's values from its own inputs (3.953, 2.674 and 0.125 kB T; 2.157, 1.185 and 0.145 kB T). Under L-b and y-1, from the 5 pF groups, pathway 1 has V0.5 -0.0749 V, lambda 3.69 kB T and forward barrier 2.65 kB T, and pathway 2 has -0.0253 V, 2.29 and 1.09 kB T: all six within tolerance. The 0.01 V groups put the pathway 2 V0.5 (-0.0217 V), lambda (2.43) and forward barrier (1.03) outside, and the per-cell groups put lambda of both pathways (3.22 and 2.54) outside. With all groups (y-0), the outcome before this revision, lambda of pathway 1 is 4.40 kB T, outside, and the outcome was partial. The reverse barriers (j-a, y-1) are 0.086 and 0.221 kB T (the paper has 0.125 and 0.145), and the forward rate ratio of pathway 2 to pathway 1 (dimer) is 67.3 (the paper has 64). Under L-c with y-1, lambda of pathway 1 is 3.25, outside. The 6.7 kB T chemical free energy was not computed.

**Result: not reproduced.** Under L-b and y-1, with C_Leq fixed at 1.098 aF, the 5 pF groups give a mean n of 1.25 (0.79 to 1.74) and a mean delta of 0.741 (0.55 to 1.17), both outside 1.7 +/- 0.17 and 0.535 +/- 0.054. The other groupings give means of 1.37 and 1.30 for n, and 0.683 and 0.686 for delta. With all groups (y-0), j-a gives n 1.63, within, and delta 0.690, outside; no range lies within the paper's under any reading. Under L-c (y-1, j-a) the means are 1.18 and 0.777.

## low-chloride-similar

> With 10 mM intracellular chloride and glutamate, results are similar to those with high
> intracellular chloride.

In [ ]:
for lab, d in [("L-b", df), ("L-c 4.82", dfB)]:
    c, n_, x_ = sets(d)
    for lim in ["y-1", "y-0"]:
        bx_ = ear1.bin_cells(x_, "CLW", 5)
        bx_ = ear1.in_range(bx_) if lim == "y-1" else bx_
        vd = ear1.fit_eq3a(bx_.dVmin, bx_.V05, bx_.n)["Vd"]
        lc = d[d.chloride_mM == 10][["orig", "pigmented", "xe991", "z", "dVmin", "C_Leq_aF", "V05"]].copy()
        lc["V0.5 - line"] = lc.V05 - (vd - lc.dVmin / 2)
        lc["C_Leq ok"] = lc.C_Leq_aF.between(0.584 - 0.065, 1.07 + 0.038)
        lc["z ok"] = lc.z.between(0.746, 0.952)
        lc["V ok"] = lc["V0.5 - line"].abs() <= 0.02
        print(lab, lim, "Vd of the XE991 line (144 mM, e-a)", round(vd, 4), "; cells meeting all three:",
              int((lc["C_Leq ok"] & lc["z ok"] & lc["V ok"]).sum()), "of 3")
        print(lc.round(4).to_string())

**Result: partially reproduced (resting on the assumed CSB rule and the assumed dVmin range).** Under L-b, all three 10 mM cells have z within 0.746 to 0.952 (0.763, 0.905, 0.921) and C_Leq within 0.519 to 1.108 aF (0.64, 0.87 and 1.09 aF). The XE991 Eq. 3a line under y-1 has Vd 0.0505 V; cells 87 and 89 lie within 0.02 V of it (residuals -0.005 and -0.017 V), cell 88 does not (-0.029 V). Two of three cells meet all three conditions, the partial tier. With all bins (y-0), the outcome before this revision, the line has Vd 0.0847 V and no cell meets the V0.5 condition. Under L-c one cell meets all three under either reading.

## recording-time

> Recordings were made 47 to 182 min after sacrifice (mean 125 +/- 32 min pigmented, 112 +/- 36 min
> albino), and the electrical parameters do not correlate with recording time.

In [ ]:
print("range", df.t_after_death_min.min(), "to", df.t_after_death_min.max(), "min")
print(df.groupby("colony").t_after_death_min.agg(["size", "mean", "std"]).round(1))
rows = []
for col in ["pigmented", "albino"]:
    g = df[df.colony == col]
    for y in ["QTm", "CLW", "V05", "alphaB", "dVmin"]:
        pr = stats.pearsonr(g.t_after_death_min, g[y])
        rows.append({"colony": col, "parameter": y, "n": len(g), "r": pr.statistic, "p": pr.pvalue, "p x 10": min(1, pr.pvalue * 10)})
print(pd.DataFrame(rows).round(4))

**Result: reproduced (resting on the assumed test).** Recordings were made 47 to 182 min after death: 124.7 +/- 31.8 min for 47 pigmented cells and 112.0 +/- 35.6 min for 42 albino cells, both within tolerance. None of the ten correlations with recording time is significant (|r| <= 0.19, p >= 0.22).

## two-hyperpolarized-states

> The relation of V0.5 to charge density, with slope near -1/2, Vd constant and Vh varying, shows
> that the prestin sensor has two hyperpolarized states and two charge-displacement pathways in the
> lateral membrane.

Judged from the claims it depends on (above).

**Result: partially reproduced.** Of the claims this rests on, fig3a-eq3a-fit is reproduced under L-b, and fig4a-vh-rises-vd-constant, fig4d-cleq-two-limits and mean-slope-half are not. The main fit's unconstrained slope is negative (-0.42) and within 10% of -0.401, Vd is constant and Vh rises monotonically without XE991, and C_Leq rises with V0.5 (0.650 to 1.048 aF, the lower value outside its tolerance by 0.0006 aF under y-1). But the mean of the three slopes is -0.407, not within 10% of -0.48, and the rise of Vh is 2.67-fold, not threefold. The criterion's not-reproduced tier needs a non-negative slope or a varying Vd, and neither holds; its reproduced tier needs every claim it rests on, which does not hold. The outcome is partial, the closest tier.

## Blocked claims

- splice-isoform-x1: the PCR and sequencing are not in the data set (data missing).
- molecular-model: the I-TASSER model is not deposited (data missing).
- fig3a-other-rodents, fig3c-engineered-mice, fig4f-all-rodents, figs7-hearing-loss,
  v05-temperature-corrected: rest on values from other papers (or a temperature correction the
  paper does not state numerically) that are not under `paper/` and were not fetched.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cohort | reproduced |
| figs1-example-cell | reproduced |
| nlw-capacitance | reproduced (assumed CSB rule) |
| clw-by-turn | reproduced (assumed CSB rule) |
| fig1-charge-not-linear | reproduced (assumed CSB rule) |
| fig1-two-state-bounds | not reproduced (assumed CSB rule) |
| alpha-pigmented-vs-albino | reproduced |
| alpha-gradient-pigmented | reproduced |
| table1-pigmented-xe991 | partially reproduced |
| table1-pigmented-no-xe991 | partially reproduced |
| table1-albino-no-gradient | reproduced |
| table1-xe991-no-effect-alpha | partially reproduced |
| figs2-alphab-vs-clw | reproduced (assumed test) |
| splice-isoform-x1 | blocked: data missing |
| v05-by-turn | reproduced |
| fig2b-v05-vs-clw | partially reproduced |
| fig2b-no-phenotype-difference | not reproduced |
| v05-gradient-length | not reproduced |
| xe991-voltage-clamp | not reproduced |
| table2-xe991-v05-shift | reproduced |
| table2-pigmented-xe991 | reproduced |
| table2-pigmented-no-xe991 | reproduced |
| table2-albino | reproduced |
| v05-not-correlated-voltage-drop | reproduced (assumed test) |
| fig3a-eq3a-fit | reproduced (assumed CSB rule) |
| fig3a-xe991-fit | reproduced (assumed CSB rule and dVmin range) |
| fig3a-high-density-deviation | reproduced (assumed CSB rule and tests) |
| figs4-fine-bins | partially reproduced (assumed CSB rule) |
| tables3-by-region | reproduced (assumed CSB rule) |
| fig3a-other-rodents | blocked (outside values) |
| fig3c-engineered-mice | blocked (outside values) |
| capacitance-asymmetry | not reproduced (assumed plateau and CSB rules) |
| figs5-specific-capacitance | reproduced (assumed plateau rule) |
| fig3b-negative-clw-fit | not reproduced |
| mean-slope-half | not reproduced (assumed CSB rule, dVmin range and readings) |
| fig4a-vh-rises-vd-constant | not reproduced |
| fig4d-cleq-two-limits | not reproduced (assumed CSB rule and dVmin range) |
| fig4c-sensor-area | not reproduced |
| fig4e-sensor-charge | reproduced (assumed CSB rule) |
| table3-pathway-energies | reproduced (assumed CSB rule and dVmin range) |
| alternative-limit | not reproduced |
| fig4f-all-rodents | blocked (outside values) |
| low-chloride-similar | partially reproduced (assumed CSB rule and dVmin range) |
| molecular-model | blocked: data missing |
| figs7-hearing-loss | blocked (outside values) |
| v05-temperature-corrected | blocked (outside values) |
| recording-time | reproduced (assumed test) |
| two-hyperpolarized-states | partially reproduced |

How the analysis differs from the paper's. The per-cell CL, QTm, V0.5 and alphaB are the values
the data set stores, which equal the Fig. S1 legend for its example cell. CLW follows the amended
shared method of the claims file: CL - CNLW with CNLW = CSB + root per cell (reading L-b). The data
set stores no CSB, so the stereocilia capacitance is computed by the rule of Corbitt et al. (2012),
the paper the method cites, on the stored cell surface area, with the 6 cells without an area given
the median CSB of their turn; this rule is an assumption, and every outcome on CLW, dVmin or C_Leq
rests on it. It gives the Fig. S1 cell CLW 18.30 pF (legend 18.56 pF) and a root of 5.10 pF. Under
L-b, against CL - 4.82 pF (L-c, shown for comparison), the Eq. 3a fit without XE991, the Eq. 6b R2
of Fig. 1 and the Fig. S4 fit come within tolerance, while the band of Fig. 1 fails. The CLW of the
smallest cells shrinks (file 57: 0.48 pF), so the two smallest XE991 cells form a bin of mean dVmin
0.80 V, and one Fig. 3b cell has a negative CLWn. Bins and groups with a mean dVmin outside (0, 0.4]
V are left out of the Eq. 3a fits and the groups built on them (an assumption from the Fig. S4
legend and the Discussion; all bins are shown as a variant). This makes the XE991 fit and Table 3
pass and the low-chloride cells pass in part, and puts the lower C_Leq limit 0.0006 aF outside its
tolerance. CLn is
not stored. Its rule (the two most negative ramp points, a plateau fraction chosen to give the
paper's 62 cells) is an assumption. Group counts differ slightly from the paper's: 27 against 28
pigmented cells without XE991, 17 against 15 or 16 pigmented XE991 cells with 144 mM chloride, and
69 against 70 cells in Fig. 2.

What was found about the data set. It holds all 89 cells, with the paper's group counts, and the
derived values the paper reports. It does not hold CLW, CLn, CSB or the selection of cells for each
table. The HDF5 `I_mean` array stores its values in column order under a row-major shape. The
stored surface area follows two formulas. One cell holds at +0.06 V where the description says
-0.06 V. The MATLAB `toxins` field labels 5 of the 20 XE991 cells `yes`. Original cell numbers are
not unique (13 are used twice).